
# 1. Environment and Experiment Setup

In this notebook, we will implement a JEPA-based model for multiple-choice decision prediction.

We will use a pretrained ModernBERT encoder as the backbone and implement the JEPA-specific components ourselves.

### Experiment scope

- Question types: set by `included_question_types` in the configuration (this run: `noul` only;
  `choice` can be added back; `score` is not supported)
- Online input: State + question + all candidates
- Target: set by `target_mode` (section 25b). This run: two fixed answer vectors, ModernBERT's encoding of
  `yes` and `no` (no state, no question), so no target encoder is needed. First design (`candidate_text`):
  the EMA target encoder on state + question + candidate
- Target encoder: EMA copy of the online encoder
- Predictor: Custom trainable network
- Inference: Candidate-wise latent similarity matching

### Hardware

The initial target environment was an NVIDIA T4 GPU with 16 GB VRAM. The full run uses a Modal L4 (24 GB).

### Configuration

Every setting of the run is in `CONFIG` in the next cell. An environment variable with the setting's
name in capitals overrides it (the Modal runner passes them), e.g. `NUM_EPOCHS=1`.

### Objectives of this cell

1. Inspect Python and PyTorch versions.
2. Detect CUDA availability.
3. Identify the available GPU.
4. Establish reproducibility settings.
5. Define experiment configuration.

We will validate the environment before proceeding to model initialization.


In [ ]:
import os
import sys
import platform
import random
import numpy as np
import torch

# --------------------------------------------------
# 1. Experiment configuration (every setting of the run)
#    An environment variable with the key in capitals overrides a value,
#    e.g. NUM_EPOCHS=1 or INCLUDED_QUESTION_TYPES=noul,choice.
# --------------------------------------------------

DEFAULTS = {
    # Reproducibility
    "seed": 42,

    # Model
    "model_name": "answerdotai/ModernBERT-base",
    "latent_dim": 256,
    "predictor_hidden_dim": 512,
    "target_ema_decay": 0.996,
    "temperature": 0.1,

    # Answer targets (section 25b): what each candidate's latent is
    #   "encoder":        ModernBERT's own encoding of the answer word (target_words), computed once, then fixed
    #   "fixed":          two opposite unit vectors (the comparison without ModernBERT's meaning)
    #   "candidate_text": the first design, the EMA target encoder on state + question + candidate
    "target_mode": "encoder",
    "target_words": ["yes", "no"],                # for the true and the false candidate
    "target_repeats": 64,                         # "yes yes yes ..." (target_check: 64 x, spaces)
    "target_separator": " ",
    "target_pooling": "mean_words",               # mean over the word tokens, [CLS]/[SEP] left out; or "mean_all"
    "target_gate_gold": 0.9,                      # the targets must let a gold this confident be reached

    # Data
    "included_question_types": ["noul"],          # "choice" can be added back
    "data_path": "data/signpost_d_decisions/train.jsonl",
    "data_url": ("https://raw.githubusercontent.com/skamalj/llms-the-hard-way/"
                 "decider-jepa/data/signpost_d_decisions/train.jsonl"),
    "probability_sum_tolerance": 0.01,
    "max_sequence_length": 1024,                  # chosen from the profiling in section 2
    "padding_strategy": "dynamic",
    "truncation": True,

    # Group-aware split
    "split_seed": 42,
    "train_ratio": 0.80,
    "validation_ratio": 0.10,
    "test_ratio": 0.10,

    # Optimisation
    "backbone_lr": 2e-5,
    "jepa_lr": 1e-4,
    "weight_decay": 0.01,
    "grad_clip_norm": 1.0,
    "batch_size": 4,
    "num_workers": 0,
    "num_epochs": 3,

    # Logging and checkpoints
    "log_interval": 50,                           # batches
    "checkpoint_interval": 250,                   # batches
    "run_dir": "signpost_jepa",                   # checkpoints/ and logs/ go here
    "run_demo_epoch": False,                      # section 28's single demo epoch (the full run is section 29)

    # Benchmark: Typed Decisions test split (scored, never trained on)
    "td_dataset": "LocalLLaMA/typed-decisions",
    "td_file": "all/test-00000-of-00001.parquet",
    "reference_model": "skamalj/signpost-d-4b",   # its public test predictions, for a reference row
    "reference_file": "typed_decisions_test_predictions.json",
}


def from_environment(key, default):
    value = os.environ.get(key.upper())
    if value is None:
        return default
    if isinstance(default, bool):
        return value.strip().lower() in ("1", "true", "yes")
    if isinstance(default, int):
        return int(value)
    if isinstance(default, float):
        return float(value)
    if isinstance(default, list):
        return [v.strip() for v in value.split(",") if v.strip()]
    return value


CONFIG = {key: from_environment(key, value) for key, value in DEFAULTS.items()}

# --------------------------------------------------
# 2. Reproducibility
# --------------------------------------------------

SEED = CONFIG["seed"]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --------------------------------------------------
# 3. Environment information
# --------------------------------------------------

print("=" * 55)
print("ENVIRONMENT")
print("=" * 55)

print(f"Python version : {sys.version}")
print(f"Platform       : {platform.platform()}")
print(f"PyTorch        : {torch.__version__}")
print(f"NumPy          : {np.__version__}")

# --------------------------------------------------
# 4. GPU detection
# --------------------------------------------------

print("\n" + "=" * 55)
print("GPU INFORMATION")
print("=" * 55)

if torch.cuda.is_available():

    DEVICE = torch.device("cuda")

    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)

        print(f"\nGPU {i}: {props.name}")
        print(f"VRAM          : {props.total_memory / (1024**3):.2f} GB")
        print(f"Compute       : {props.major}.{props.minor}")
        print(f"CUDA          : {torch.version.cuda}")

    print(f"\nActive device : {DEVICE}")

else:
    DEVICE = torch.device("cpu")
    print("CUDA unavailable. Using CPU.")

CONFIG["device"] = DEVICE

print("\n" + "=" * 55)
print("EXPERIMENT CONFIGURATION")
print("=" * 55)

for key, value in CONFIG.items():
    overridden = " (environment)" if key.upper() in os.environ else ""
    print(f"{key:30s}: {value}{overridden}")

# --------------------------------------------------
# 5. Basic validation
# --------------------------------------------------

assert torch.__version__ is not None
assert CONFIG["included_question_types"], "No question types selected."
assert set(CONFIG["included_question_types"]) <= {"noul", "choice"}, (
    "Only noul and choice are supported."
)
assert CONFIG["target_mode"] in ("encoder", "fixed", "candidate_text")
assert CONFIG["target_mode"] == "candidate_text" or CONFIG["included_question_types"] == ["noul"], (
    "Answer targets are defined for NOUL (true / false) only."
)
assert abs(
    CONFIG["train_ratio"] + CONFIG["validation_ratio"] + CONFIG["test_ratio"] - 1.0
) < 1e-9

print("\nEnvironment setup completed successfully.")



# 2. Tokenizer Initialization and Context-Length Profiling

Our input contains potentially large nested JSON states, question instructions and candidate descriptions.

We must determine the required context length empirically before selecting a training configuration.

### What this cell does

1. Load the pretrained ModernBERT tokenizer.
2. Read the original JSONL dataset.
3. Expand each state into individual questions.
4. Retain only `noul` and `choice` questions.
5. Construct the complete online-branch input for each question.
6. Tokenize without truncation.
7. Calculate:
   - Minimum and maximum token counts
   - Mean and median
   - P90, P95 and P99
   - Percentage exceeding 512, 1024, 2048, 4096 and 8192 tokens
8. Report statistics separately for NOUL and Choice.

### Important

We are deliberately not setting `max_length` or enabling truncation.

We want to see the actual context requirements before deciding whether to use 2048, 4096 or 8192 tokens.

The dataset path below must point to the complete JSONL dataset.


In [ ]:

import json
import numpy as np
import pandas as pd

from pathlib import Path
from transformers import AutoTokenizer
from tqdm.auto import tqdm

# --------------------------------------------------
# 1. Dataset and tokenizer configuration
# --------------------------------------------------

DATA_PATH = Path(CONFIG["data_path"])

MODEL_NAME = CONFIG["model_name"]

DATA_URL = CONFIG["data_url"]

if not DATA_PATH.exists():
    import urllib.request
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {DATA_URL}")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH.resolve()}"

print(f"Dataset: {DATA_PATH.resolve()}")
print(f"Tokenizer: {MODEL_NAME}")

# --------------------------------------------------
# 2. Load ModernBERT tokenizer
# --------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("\nTokenizer information")
print("-" * 50)
print(f"Tokenizer class       : {tokenizer.__class__.__name__}")
print(f"Vocabulary size       : {tokenizer.vocab_size}")
print(f"Model maximum length  : {tokenizer.model_max_length}")
print(f"Special tokens        : {tokenizer.special_tokens_map}")

# --------------------------------------------------
# 3. Construct candidate descriptions
# --------------------------------------------------

def get_candidates(question):

    question_type = question.get("type", "").lower()
    criteria = question.get("criteria", {})

    if question_type == "noul":
        # Some NOUL questions have no criteria (null): the statement
        # itself is judged true or false.
        if criteria is None:
            criteria = {
                "true": "The statement is true.",
                "false": "The statement is false."
            }
        if not isinstance(criteria, dict):
            raise ValueError("NOUL criteria must be a dictionary.")

        return [
            {
                "key": str(key),
                "description": str(value)
            }
            for key, value in criteria.items()
        ]

    if question_type == "choice":
        if not isinstance(criteria, dict):
            raise ValueError("Choice criteria must be a dictionary.")

        return [
            {
                "key": str(key),
                "description": str(value)
            }
            for key, value in criteria.items()
        ]

    return []


# --------------------------------------------------
# 4. Construct complete online input
# --------------------------------------------------

def build_online_input(record, question_id, question):

    payload = {
        "domain": record.get("domain"),
        "state": record.get("state"),
        "question": {
            "id": question_id,
            "type": question.get("type"),
            "instructions": question.get("instructions"),
            "candidates": get_candidates(question)
        }
    }

    return json.dumps(
        payload,
        ensure_ascii=False,
        separators=(",", ":")
    )


# --------------------------------------------------
# 5. Read JSONL and expand questions
# --------------------------------------------------

samples = []
invalid_records = []

with DATA_PATH.open("r", encoding="utf-8") as file:

    for line_number, line in enumerate(file, start=1):

        line = line.strip()

        if not line:
            continue

        try:
            record = json.loads(line)

            questions = record.get("questions", {})

            for question_id, question in questions.items():

                question_type = question.get("type", "").lower()

                if question_type not in CONFIG["included_question_types"]:
                    continue

                candidates = get_candidates(question)

                if not candidates:
                    continue

                samples.append({
                    "record_id": record.get("id"),
                    "state_id": record.get("state", {}).get("record_id"),
                    "question_id": question_id,
                    "question_type": question_type,
                    "candidate_count": len(candidates),
                    "text": build_online_input(
                        record,
                        question_id,
                        question
                    )
                })

        except Exception as exc:
            invalid_records.append({
                "line": line_number,
                "error": str(exc)
            })

print("\nDataset expansion")
print("-" * 50)
print(f"Valid question samples : {len(samples)}")
print(f"Invalid JSONL records  : {len(invalid_records)}")

if invalid_records:
    print("\nFirst invalid records:")
    for item in invalid_records[:5]:
        print(item)

assert len(samples) > 0, f"No {CONFIG['included_question_types']} questions found."

# --------------------------------------------------
# 6. Tokenize without truncation
# --------------------------------------------------

token_counts = []

BATCH_SIZE = 128

for start in tqdm(
    range(0, len(samples), BATCH_SIZE),
    desc="Tokenizing samples"
):

    batch = samples[start:start + BATCH_SIZE]
    texts = [sample["text"] for sample in batch]

    encoded = tokenizer(
        texts,
        add_special_tokens=True,
        truncation=False,
        padding=False
    )

    for sample, input_ids in zip(batch, encoded["input_ids"]):
        token_counts.append({
            "record_id": sample["record_id"],
            "state_id": sample["state_id"],
            "question_id": sample["question_id"],
            "question_type": sample["question_type"],
            "candidate_count": sample["candidate_count"],
            "token_count": len(input_ids)
        })

# --------------------------------------------------
# 7. Build profiling DataFrame
# --------------------------------------------------

profile_df = pd.DataFrame(token_counts)

print("\nQuestion type distribution")
print("-" * 50)

print(
    profile_df["question_type"]
    .value_counts()
    .to_string()
)

# --------------------------------------------------
# 8. Token length statistics
# --------------------------------------------------

def calculate_statistics(df):

    lengths = df["token_count"].to_numpy()

    return {
        "samples": len(lengths),
        "min": int(np.min(lengths)),
        "mean": round(float(np.mean(lengths)), 1),
        "median": round(float(np.median(lengths)), 1),
        "p90": round(float(np.percentile(lengths, 90)), 1),
        "p95": round(float(np.percentile(lengths, 95)), 1),
        "p99": round(float(np.percentile(lengths, 99)), 1),
        "max": int(np.max(lengths))
    }


print("\nToken length statistics")
print("=" * 75)

overall_stats = calculate_statistics(profile_df)

print("\nALL INCLUDED QUESTIONS")
for key, value in overall_stats.items():
    print(f"{key:15s}: {value}")

for question_type in CONFIG["included_question_types"]:

    subset = profile_df[
        profile_df["question_type"] == question_type
    ]

    if len(subset) == 0:
        continue

    stats = calculate_statistics(subset)

    print(f"\n{question_type.upper()}")

    for key, value in stats.items():
        print(f"{key:15s}: {value}")

# --------------------------------------------------
# 9. Truncation risk analysis
# --------------------------------------------------

THRESHOLDS = [512, 1024, 2048, 4096, 8192]

print("\nContext length coverage")
print("=" * 75)

coverage_rows = []

for question_type in ["ALL", *CONFIG["included_question_types"]]:

    if question_type == "ALL":
        subset = profile_df
    else:
        subset = profile_df[
            profile_df["question_type"] == question_type
        ]

    for threshold in THRESHOLDS:

        exceeded = (subset["token_count"] > threshold).sum()

        coverage_rows.append({
            "question_type": question_type.upper(),
            "max_length": threshold,
            "exceeded_count": int(exceeded),
            "exceeded_pct": round(
                100 * exceeded / len(subset), 2
            )
        })

coverage_df = pd.DataFrame(coverage_rows)

print(
    coverage_df.pivot(
        index="max_length",
        columns="question_type",
        values="exceeded_pct"
    ).to_string()
)

# --------------------------------------------------
# 10. Inspect longest samples
# --------------------------------------------------

print("\n20 longest question inputs")
print("=" * 75)

display(
    profile_df.nlargest(20, "token_count")[
        [
            "record_id",
            "state_id",
            "question_id",
            "question_type",
            "candidate_count",
            "token_count"
        ]
    ].reset_index(drop=True)
)

print("\nContext profiling completed.")


In [ ]:
# The context length chosen from the profiling above is set in CONFIG (cell 1).
print("max_sequence_length:", CONFIG["max_sequence_length"])
print("padding_strategy   :", CONFIG["padding_strategy"])
print("truncation         :", CONFIG["truncation"])



# 3. Dataset Integrity and Gold-Label Validation

We validate the complete dataset before preparing JEPA training examples.

### Supported question formats

**NOUL with explicit criteria**

The question provides candidate descriptions for `true` and `false`.

**NOUL with null criteria**

The candidate set is implicitly:
- true
- false

We use the literal candidate labels as their descriptions. We do not invent additional descriptions.

**Choice**

Candidates are extracted from the question's criteria dictionary. Each dictionary key is the candidate identifier and its value is the candidate description.

### Gold-label validation

For every included question:
- Gold entry must exist.
- Gold type must match question type.
- Gold candidate keys must match available candidate keys.
- Probabilities must be numeric, finite and non-negative.
- Probability distributions must sum to 1 within tolerance.

We preserve the original probability distributions without converting them into hard labels.

### Identifier handling

- `record_id`: top-level dataset identifier.
- `state_record_id`: optional identifier inside the state.
- `state_group_id`: top-level record ID, used for dataset splitting.

All questions belonging to the same top-level record will remain in the same dataset split.

### Expected result

For the current dataset:
- 2,575 records.
- 3,908 NOUL questions.
- 3,604 Choice questions.
- 7,512 valid training examples.
- 4,762 Score questions excluded.


In [ ]:

import json
import math
import pandas as pd

from collections import Counter
from pathlib import Path

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

ALLOWED_TYPES = set(CONFIG["included_question_types"])
PROBABILITY_SUM_TOLERANCE = CONFIG["probability_sum_tolerance"]

validated_samples = []
invalid_samples = []

excluded_questions = Counter()

seen_record_ids = set()
seen_question_keys = set()

duplicate_record_ids = []
duplicate_question_keys = []

# --------------------------------------------------
# 2. Candidate construction
# --------------------------------------------------

def get_candidates(question):

    question_type = str(
        question.get("type", "")
    ).lower()

    criteria = question.get("criteria")

    # NOUL with implicit binary candidates
    if question_type == "noul" and criteria is None:

        return [
            {
                "key": "true",
                "description": "true"
            },
            {
                "key": "false",
                "description": "false"
            }
        ]

    # Explicit candidate definitions
    if isinstance(criteria, dict) and criteria:

        return [
            {
                "key": str(key),
                "description": str(description)
            }
            for key, description in criteria.items()
        ]

    raise ValueError(
        f"Invalid candidate criteria for {question_type}"
    )


# --------------------------------------------------
# 3. Probability validation
# --------------------------------------------------

def validate_probabilities(probabilities):

    if not isinstance(probabilities, dict) or not probabilities:
        raise ValueError("Missing or invalid gold probabilities")

    converted = {}

    for key, value in probabilities.items():

        if (
            isinstance(value, bool)
            or not isinstance(value, (int, float))
            or not math.isfinite(value)
            or value < 0
        ):
            raise ValueError(
                f"Invalid probability for {key}: {value}"
            )

        converted[str(key)] = float(value)

    total = sum(converted.values())

    if abs(total - 1.0) > PROBABILITY_SUM_TOLERANCE:
        raise ValueError(
            f"Gold probability sum is {total:.6f}, expected 1.0"
        )

    return converted


# --------------------------------------------------
# 4. Validate one question
# --------------------------------------------------

def validate_question(record, question_id, question):

    errors = []

    record_id = record.get("id")
    state = record.get("state")

    question_type = str(
        question.get("type", "")
    ).lower()

    # Top-level record ID is the authoritative grouping ID.
    state_group_id = str(record_id) if record_id else None

    state_record_id = (
        state.get("record_id")
        if isinstance(state, dict)
        else None
    )

    # Identifiers
    if not record_id:
        errors.append("Missing top-level record ID")

    if not isinstance(state, dict) or not state:
        errors.append("Missing or invalid state")

    # Question instructions
    instructions = question.get("instructions")

    if not isinstance(instructions, str) or not instructions.strip():
        errors.append("Missing question instructions")

    # Candidates
    try:
        candidates = get_candidates(question)

        if any(
            not candidate["description"].strip()
            for candidate in candidates
        ):
            errors.append("Empty candidate description")

    except ValueError as exc:
        candidates = []
        errors.append(str(exc))

    # Gold
    gold_entry = (
        record.get("gold", {}).get(question_id)
        if isinstance(record.get("gold"), dict)
        else None
    )

    gold_probabilities = None

    if not isinstance(gold_entry, dict):

        errors.append("Missing gold entry")

    else:

        gold_type = str(
            gold_entry.get("type", "")
        ).lower()

        if gold_type != question_type:
            errors.append("Gold question type mismatch")

        try:

            gold_probabilities = validate_probabilities(
                gold_entry.get("probabilities")
            )

            candidate_keys = {
                candidate["key"]
                for candidate in candidates
            }

            gold_keys = set(gold_probabilities.keys())

            if candidate_keys != gold_keys:

                errors.append(
                    f"Candidate/gold key mismatch: "
                    f"candidates={candidate_keys}, gold={gold_keys}"
                )

        except ValueError as exc:
            errors.append(str(exc))

    return {
        "record_id": record_id,
        "state_group_id": state_group_id,
        "state_record_id": state_record_id,
        "question_id": question_id,
        "question_type": question_type,
        "instructions": instructions,
        "state": state,
        "candidates": candidates,
        "candidate_count": len(candidates),
        "gold_probabilities": gold_probabilities,
        "errors": errors
    }


# --------------------------------------------------
# 5. Read and validate JSONL
# --------------------------------------------------

total_records = 0
total_questions = Counter()

with DATA_PATH.open("r", encoding="utf-8") as file:

    for line_number, line in enumerate(file, start=1):

        if not line.strip():
            continue

        try:
            record = json.loads(line)

        except json.JSONDecodeError as exc:

            invalid_samples.append({
                "line_number": line_number,
                "record_id": None,
                "question_id": None,
                "errors": [f"Invalid JSON: {exc}"]
            })

            continue

        total_records += 1

        record_id = record.get("id")

        if record_id:

            record_id = str(record_id)

            if record_id in seen_record_ids:
                duplicate_record_ids.append(record_id)

            seen_record_ids.add(record_id)

        questions = record.get("questions")

        if not isinstance(questions, dict):

            invalid_samples.append({
                "line_number": line_number,
                "record_id": record_id,
                "question_id": None,
                "errors": ["Missing or invalid questions object"]
            })

            continue

        for question_id, question in questions.items():

            if not isinstance(question, dict):

                invalid_samples.append({
                    "line_number": line_number,
                    "record_id": record_id,
                    "question_id": question_id,
                    "errors": ["Invalid question object"]
                })

                continue

            question_type = str(
                question.get("type", "")
            ).lower()

            total_questions[question_type] += 1

            # Exclude Score and unsupported types
            if question_type not in ALLOWED_TYPES:

                excluded_questions[
                    question_type or "missing_type"
                ] += 1

                continue

            result = validate_question(
                record,
                question_id,
                question
            )

            # Duplicate detection uses top-level record ID.
            sample_key = (
                result["record_id"],
                str(question_id)
            )

            if sample_key in seen_question_keys:

                duplicate_question_keys.append(sample_key)

                result["errors"].append(
                    "Duplicate record/question combination"
                )

            seen_question_keys.add(sample_key)

            if result["errors"]:

                invalid_samples.append({
                    "line_number": line_number,
                    "record_id": record_id,
                    "question_id": question_id,
                    "errors": result["errors"]
                })

            else:

                validated_samples.append(result)


# --------------------------------------------------
# 6. Create report DataFrames
# --------------------------------------------------

valid_df = pd.DataFrame([
    {
        "record_id": s["record_id"],
        "state_group_id": s["state_group_id"],
        "state_record_id": s["state_record_id"],
        "question_id": s["question_id"],
        "question_type": s["question_type"],
        "candidate_count": s["candidate_count"],
        "candidates": s["candidates"],
        "gold_probabilities": s["gold_probabilities"]
    }
    for s in validated_samples
])

invalid_df = pd.DataFrame(invalid_samples)


# --------------------------------------------------
# 7. Dataset integrity report
# --------------------------------------------------

print("=" * 65)
print("DATASET INTEGRITY REPORT")
print("=" * 65)

print(f"Total records              : {total_records}")
print(f"Total questions            : {sum(total_questions.values())}")
print(f"Valid examples             : {len(validated_samples)}")
print(f"Invalid examples           : {len(invalid_samples)}")

print(f"Unique record IDs          : {len(seen_record_ids)}")
print(f"Duplicate record IDs       : {len(duplicate_record_ids)}")
print(f"Duplicate question keys    : {len(duplicate_question_keys)}")

print("\nQuestion type distribution")
print("-" * 65)

for question_type, count in total_questions.items():
    print(f"{question_type:20s}: {count}")

print("\nExcluded question types")
print("-" * 65)

for question_type, count in excluded_questions.items():
    print(f"{question_type:20s}: {count}")


# --------------------------------------------------
# 8. Valid question distribution
# --------------------------------------------------

print("\nValid question type distribution")
print("-" * 65)

print(
    valid_df["question_type"]
    .value_counts()
    .to_string()
)

print("\nCandidate count distribution")
print("-" * 65)

print(
    pd.crosstab(
        valid_df["question_type"],
        valid_df["candidate_count"]
    ).to_string()
)


# --------------------------------------------------
# 9. State identifier coverage
# --------------------------------------------------

print("\nState identifier coverage")
print("-" * 65)

print(
    f"Examples with state.record_id: "
    f"{valid_df['state_record_id'].notna().sum()}"
)

print(
    f"Examples without state.record_id: "
    f"{valid_df['state_record_id'].isna().sum()}"
)

print(
    f"Unique split groups: "
    f"{valid_df['state_group_id'].nunique()}"
)


# --------------------------------------------------
# 10. Inspect invalid examples
# --------------------------------------------------

if not invalid_df.empty:

    print("\nFirst 20 invalid examples")
    display(invalid_df.head(20))

else:

    print("\nNo invalid examples found.")


# --------------------------------------------------
# 11. Final validation gates
# --------------------------------------------------

assert len(validated_samples) > 0
assert len(invalid_samples) == 0, (
    "Invalid examples found. Inspect invalid_df."
)

assert len(duplicate_record_ids) == 0
assert len(duplicate_question_keys) == 0

assert all(
    sample["question_type"] in ALLOWED_TYPES
    for sample in validated_samples
)

assert all(
    sample["state_group_id"]
    for sample in validated_samples
)

print("\n" + "=" * 65)
print("ALL DATASET VALIDATION GATES PASSED")
print("=" * 65)

print(f"Validated training examples: {len(validated_samples)}")
print(f"Unique split groups: {valid_df['state_group_id'].nunique()}")



# 4. Gold Distribution Structure Analysis

We have established that the dataset contains substantial soft-label supervision.

This analysis examines the structure of each gold probability distribution.

For every question, we calculate:

1. Maximum and minimum candidate probabilities.
2. Number of non-zero candidates.
3. Number of candidates tied for the maximum probability.
4. Whether the distribution is one-hot.
5. Whether the distribution is uniform.
6. Whether the distribution contains multiple non-zero probabilities.

We perform the analysis separately for NOUL and Choice.

The original gold probability distributions remain unchanged.

This is still a data-analysis step. No training labels are modified.


In [ ]:

import numpy as np
import pandas as pd

from IPython.display import display

# --------------------------------------------------
# 1. Analyze each gold probability distribution
# --------------------------------------------------

distribution_analysis = []

TOLERANCE = 1e-6

for sample in validated_samples:

    probabilities = sample["gold_probabilities"]

    values = np.array(
        list(probabilities.values()),
        dtype=np.float64
    )

    max_probability = float(values.max())
    min_probability = float(values.min())

    nonzero_count = int(
        np.sum(values > TOLERANCE)
    )

    tied_max_count = int(
        np.sum(
            np.isclose(
                values,
                max_probability,
                atol=TOLERANCE
            )
        )
    )

    is_one_hot = (
        np.isclose(max_probability, 1.0, atol=TOLERANCE)
        and nonzero_count == 1
    )

    is_uniform = bool(
        np.allclose(
            values,
            np.full(len(values), 1.0 / len(values)),
            atol=TOLERANCE
        )
    )

    distribution_analysis.append({

        "record_id": sample["record_id"],
        "question_id": sample["question_id"],
        "question_type": sample["question_type"],

        "candidate_count": len(values),

        "max_probability": max_probability,
        "min_probability": min_probability,

        "nonzero_candidate_count": nonzero_count,
        "tied_max_count": tied_max_count,

        "is_one_hot": is_one_hot,
        "is_uniform": is_uniform,

        "has_multiple_nonzero": nonzero_count > 1,

        "gold_probabilities": probabilities
    })


distribution_df = pd.DataFrame(distribution_analysis)

assert len(distribution_df) == len(validated_samples)

# --------------------------------------------------
# 2. Distribution structure by question type
# --------------------------------------------------

print("=" * 70)
print("GOLD DISTRIBUTION STRUCTURE")
print("=" * 70)

structure_columns = [
    "is_one_hot",
    "is_uniform",
    "has_multiple_nonzero"
]

for column in structure_columns:

    print(f"\n{column.upper()}")

    summary = (
        distribution_df
        .groupby("question_type")[column]
        .agg(
            count="sum",
            total="count"
        )
    )

    summary["percentage"] = (
        summary["count"] / summary["total"] * 100
    )

    display(summary.round(2))


# --------------------------------------------------
# 3. Tied maximum probabilities
# --------------------------------------------------

print("\nTIED MAXIMUM CANDIDATES")
print("-" * 70)

tied_summary = (
    distribution_df
    .groupby("question_type")
    .agg(
        total_examples=("question_id", "count"),
        examples_with_tied_max=(
            "tied_max_count",
            lambda x: int((x > 1).sum())
        )
    )
)

tied_summary["percentage"] = (
    tied_summary["examples_with_tied_max"]
    / tied_summary["total_examples"]
    * 100
)

display(tied_summary.round(2))


# --------------------------------------------------
# 4. Number of non-zero candidate probabilities
# --------------------------------------------------

print("\nNON-ZERO CANDIDATE COUNT")
print("-" * 70)

display(
    pd.crosstab(
        distribution_df["question_type"],
        distribution_df["nonzero_candidate_count"],
        margins=True
    )
)


# --------------------------------------------------
# 5. Probability range statistics
# --------------------------------------------------

print("\nPROBABILITY RANGE")
print("-" * 70)

distribution_df["probability_range"] = (
    distribution_df["max_probability"]
    - distribution_df["min_probability"]
)

display(
    distribution_df
    .groupby("question_type")[
        ["max_probability", "min_probability", "probability_range"]
    ]
    .agg(["mean", "median", "min", "max"])
)


# --------------------------------------------------
# 6. Inspect uniform distributions
# --------------------------------------------------

print("\nUNIFORM DISTRIBUTION EXAMPLES")
print("-" * 70)

uniform_examples = distribution_df[
    distribution_df["is_uniform"]
]

print(f"Total uniform examples: {len(uniform_examples)}")

display(
    uniform_examples[
        [
            "record_id",
            "question_id",
            "question_type",
            "candidate_count",
            "gold_probabilities"
        ]
    ].head(30)
)


# --------------------------------------------------
# 7. Inspect one-hot distributions
# --------------------------------------------------

print("\nONE-HOT DISTRIBUTION EXAMPLES")
print("-" * 70)

one_hot_examples = distribution_df[
    distribution_df["is_one_hot"]
]

print(f"Total one-hot examples: {len(one_hot_examples)}")

display(
    one_hot_examples[
        [
            "record_id",
            "question_id",
            "question_type",
            "candidate_count",
            "gold_probabilities"
        ]
    ].head(20)
)


# --------------------------------------------------
# 8. Final validation
# --------------------------------------------------

assert distribution_df["is_one_hot"].notna().all()
assert distribution_df["is_uniform"].notna().all()

print("\n" + "=" * 70)
print("DISTRIBUTION STRUCTURE ANALYSIS COMPLETED")
print("=" * 70)

print(f"Total examples: {len(distribution_df)}")
print(f"One-hot examples: {distribution_df['is_one_hot'].sum()}")
print(f"Uniform examples: {distribution_df['is_uniform'].sum()}")



# 5. Group-Aware Train / Validation / Test Split

We have 7,512 validated question examples originating from 2,552 unique state groups.

A single state can contain multiple questions.

For example:

- State A → Question 1
- State A → Question 2
- State A → Question 3
- State B → Question 1
- State B → Question 2

We must ensure that all questions belonging to State A remain in the same dataset split.

Otherwise, the model could see the same underlying state during training and testing. This would cause data leakage and produce misleading evaluation results.

### Split strategy

We will use:

- Training: 80% of state groups
- Validation: 10% of state groups
- Testing: 10% of state groups

The split is deterministic using a fixed random seed.

### Important design decisions

1. Split by the top-level record ID (`state_group_id`), not by individual questions.
2. Keep all questions from a state together.
3. Preserve the original gold probability distributions.
4. Do not use the test set during training or hyperparameter selection.
5. Report NOUL and Choice distributions separately in each split.

This cell only creates the splits. It does not tokenize or modify the examples.


In [ ]:

import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

SPLIT_SEED = CONFIG["split_seed"]

TRAIN_RATIO = CONFIG["train_ratio"]
VALIDATION_RATIO = CONFIG["validation_ratio"]
TEST_RATIO = CONFIG["test_ratio"]

assert abs(
    TRAIN_RATIO + VALIDATION_RATIO + TEST_RATIO - 1.0
) < 1e-9


# --------------------------------------------------
# 2. Identify unique state groups
# --------------------------------------------------

assert "state_group_id" in valid_df.columns
assert valid_df["state_group_id"].notna().all()

all_groups = valid_df[
    "state_group_id"
].drop_duplicates().to_numpy()

number_of_groups = len(all_groups)

print("=" * 70)
print("GROUP-AWARE DATASET SPLITTING")
print("=" * 70)

print(f"Total examples     : {len(valid_df)}")
print(f"Unique state groups: {number_of_groups}")


# --------------------------------------------------
# 3. Deterministic group-level split
# --------------------------------------------------

rng = np.random.default_rng(SPLIT_SEED)

shuffled_groups = rng.permutation(all_groups)

n_train = int(number_of_groups * TRAIN_RATIO)
n_validation = int(number_of_groups * VALIDATION_RATIO)

train_groups = set(
    shuffled_groups[:n_train]
)

validation_groups = set(
    shuffled_groups[n_train:n_train + n_validation]
)

test_groups = set(
    shuffled_groups[n_train + n_validation:]
)


# --------------------------------------------------
# 4. Verify group separation
# --------------------------------------------------

assert train_groups.isdisjoint(validation_groups)
assert train_groups.isdisjoint(test_groups)
assert validation_groups.isdisjoint(test_groups)

assert (
    len(train_groups)
    + len(validation_groups)
    + len(test_groups)
    == number_of_groups
)


# --------------------------------------------------
# 5. Create DataFrame splits
# --------------------------------------------------

train_df = valid_df[
    valid_df["state_group_id"].isin(train_groups)
].copy()

validation_df = valid_df[
    valid_df["state_group_id"].isin(validation_groups)
].copy()

test_df = valid_df[
    valid_df["state_group_id"].isin(test_groups)
].copy()


# --------------------------------------------------
# 6. Create sample splits
# --------------------------------------------------

train_samples = [
    sample for sample in validated_samples
    if sample["state_group_id"] in train_groups
]

validation_samples = [
    sample for sample in validated_samples
    if sample["state_group_id"] in validation_groups
]

test_samples = [
    sample for sample in validated_samples
    if sample["state_group_id"] in test_groups
]


# --------------------------------------------------
# 7. Verify example counts
# --------------------------------------------------

assert len(train_samples) == len(train_df)
assert len(validation_samples) == len(validation_df)
assert len(test_samples) == len(test_df)

assert (
    len(train_samples)
    + len(validation_samples)
    + len(test_samples)
    == len(validated_samples)
)


# --------------------------------------------------
# 8. Split summary
# --------------------------------------------------

split_summary = pd.DataFrame({

    "split": [
        "train",
        "validation",
        "test"
    ],

    "state_groups": [
        len(train_groups),
        len(validation_groups),
        len(test_groups)
    ],

    "examples": [
        len(train_samples),
        len(validation_samples),
        len(test_samples)
    ]
})

split_summary["example_percentage"] = (
    split_summary["examples"]
    / len(validated_samples)
    * 100
)

display(split_summary.round(2))


# --------------------------------------------------
# 9. Question type distribution per split
# --------------------------------------------------

print("\nQUESTION TYPE DISTRIBUTION")
print("-" * 70)

type_distribution = pd.concat({

    "train": train_df["question_type"].value_counts(),

    "validation": validation_df["question_type"].value_counts(),

    "test": test_df["question_type"].value_counts()

}, axis=1).fillna(0).astype(int)

display(type_distribution)


# --------------------------------------------------
# 10. Verify no state-group leakage
# --------------------------------------------------

train_ids = set(train_df["state_group_id"])
validation_ids = set(validation_df["state_group_id"])
test_ids = set(test_df["state_group_id"])

assert train_ids.isdisjoint(validation_ids)
assert train_ids.isdisjoint(test_ids)
assert validation_ids.isdisjoint(test_ids)

print("\nLEAKAGE CHECK")
print("-" * 70)

print("Train vs Validation : No overlap")
print("Train vs Test       : No overlap")
print("Validation vs Test  : No overlap")


# --------------------------------------------------
# 11. Final checks
# --------------------------------------------------

assert len(train_samples) > 0
assert len(validation_samples) > 0
assert len(test_samples) > 0

print("\n" + "=" * 70)
print("DATASET SPLITTING COMPLETED")
print("=" * 70)

print(f"Training examples  : {len(train_samples)}")
print(f"Validation examples: {len(validation_samples)}")
print(f"Test examples      : {len(test_samples)}")

print("\nAll split integrity checks passed.")



# 6. Implementing the JEPA Candidate Distribution Loss

In this cell, we implement and test the mathematical loss that compares the predicted latent representation against a variable number of candidate target representations.

The implementation performs four operations:

### 1. Normalize latent representations

Normalize the predicted latent and each candidate latent using L2 normalization.

This allows us to compare their directions rather than their magnitudes.

### 2. Calculate cosine distances

Calculate the cosine distance between the predicted latent and every candidate target latent.

A smaller distance indicates a closer match.

### 3. Convert distances into probabilities

Apply softmax to the negative distances, divided by a temperature parameter (initially 0.1).

This produces a probability distribution over the available candidates.

### 4. Calculate KL divergence

Compare the predicted probability distribution against the gold distribution using:

KL(P || Q)

Where P is the dataset's gold distribution and Q is the model's predicted distribution.

The loss measures how different these two distributions are.

### 5. Test dynamic candidate counts

Finally, test the same function with 2, 4, 5 and 26 candidates.

The number of candidates must not be hardcoded into the loss function.

### Expected outcome

The function should return:
- KL divergence loss.
- Predicted candidate probabilities.
- Candidate distances.
- Candidate logits.

The target representations are detached from gradient computation.

This is a standalone mathematical test using synthetic latent vectors, not actual ModernBERT outputs.


In [ ]:

import torch
import torch.nn.functional as F

def jepa_candidate_distribution_loss(
    predicted_latent,
    candidate_target_latents,
    gold_probabilities,
    temperature=CONFIG["temperature"],
):
    """
    Supports:
      Single sample:
        predicted:  [D]
        candidates: [N, D]
        gold:       [N]

      Batch:
        predicted:  [B, D]
        candidates: [B, N, D]
        gold:       [B, N]
    """

    single_sample = predicted_latent.ndim == 1

    if single_sample:
        predicted_latent = predicted_latent.unsqueeze(0)
        candidate_target_latents = candidate_target_latents.unsqueeze(0)
        gold_probabilities = gold_probabilities.unsqueeze(0)

    assert predicted_latent.ndim == 2
    assert candidate_target_latents.ndim == 3
    assert gold_probabilities.ndim == 2

    batch_size, num_candidates, latent_dim = candidate_target_latents.shape

    assert predicted_latent.shape == (batch_size, latent_dim)
    assert gold_probabilities.shape == (batch_size, num_candidates)
    assert temperature > 0

    assert torch.isfinite(predicted_latent).all()
    assert torch.isfinite(candidate_target_latents).all()
    assert torch.isfinite(gold_probabilities).all()

    assert (gold_probabilities >= 0).all()

    gold_sums = gold_probabilities.sum(dim=-1, keepdim=True)
    assert torch.allclose(
        gold_sums,
        torch.ones_like(gold_sums),
        atol=0.01,
    )

    # Normalize latent vectors
    predicted_norm = F.normalize(predicted_latent, dim=-1)
    candidate_norm = F.normalize(candidate_target_latents, dim=-1)

    # Stop gradients through target representations
    candidate_norm = candidate_norm.detach()

    # Cosine distance: [B, N]
    distances = 1.0 - torch.einsum(
        "bd,bnd->bn",
        predicted_norm,
        candidate_norm,
    )

    # Candidate probability distribution
    logits = -distances / temperature
    log_predicted_probs = F.log_softmax(logits, dim=-1)
    predicted_probs = log_predicted_probs.exp()

    # KL divergence: gold || predicted
    safe_gold = gold_probabilities.clamp_min(1e-12)

    kl_per_candidate = torch.where(
        gold_probabilities > 0,
        gold_probabilities * (
            safe_gold.log() - log_predicted_probs
        ),
        torch.zeros_like(gold_probabilities),
    )

    loss_per_sample = kl_per_candidate.sum(dim=-1)
    loss = loss_per_sample.mean()

    if single_sample:
        return (
            loss,
            predicted_probs.squeeze(0),
            distances.squeeze(0),
            logits.squeeze(0),
        )

    return loss, predicted_probs, distances, logits



# 7. Temperature Sensitivity Analysis

In this cell, we investigate how the softmax temperature affects the conversion of latent distances into candidate probabilities.

We will use the same synthetic predicted latent, candidate target latents and gold probability distribution from Cell 6.

We will evaluate five temperatures:

- 0.05
- 0.10
- 0.20
- 0.50
- 1.00

For each temperature, we will:

1. Calculate candidate probabilities using softmax over negative cosine distances.
2. Calculate KL divergence against the gold distribution.
3. Compare the predicted probability assigned to each candidate.
4. Identify which temperature produces the lowest KL loss.

### What to observe

A lower temperature makes the distribution sharper. A higher temperature makes it flatter.

We want to understand how this affects the model's ability to reproduce the gold distribution.

The lowest KL loss in this synthetic example identifies the best temperature for these particular latent distances only. It does not establish the optimal temperature for actual ModernBERT training.

No model parameters are trained or updated in this experiment.


In [ ]:

import torch
import torch.nn.functional as F
import pandas as pd

from IPython.display import display

# --------------------------------------------------
# 1. Define temperature values
# --------------------------------------------------

temperatures = [0.05, 0.10, 0.20, 0.50, 1.00]

# --------------------------------------------------
# 2. Reuse synthetic data from Cell 6
# --------------------------------------------------

z_pred = torch.tensor([1.0, 0.0])

candidate_latents = torch.tensor([
    [1.0, 0.0],     # A
    [0.8, 0.6],     # B
    [0.0, 1.0],     # C
    [-1.0, 0.0]     # D
])

gold = torch.tensor([0.50, 0.30, 0.15, 0.05])

candidate_names = ["A", "B", "C", "D"]

# --------------------------------------------------
# 3. Calculate cosine distances once
# --------------------------------------------------

z_pred_normalized = F.normalize(z_pred, p=2, dim=0)

candidate_normalized = F.normalize(
    candidate_latents,
    p=2,
    dim=1
)

distances = 1.0 - torch.matmul(
    candidate_normalized,
    z_pred_normalized
)

print("Candidate cosine distances:")
for name, distance in zip(candidate_names, distances):
    print(f"{name}: {distance.item():.4f}")

# --------------------------------------------------
# 4. Evaluate each temperature
# --------------------------------------------------

results = []
probability_rows = []

for temperature in temperatures:

    logits = -distances / temperature

    log_probs = F.log_softmax(logits, dim=0)

    predicted_probs = log_probs.exp()

    # KL(P || Q)
    kl_loss = torch.sum(
        gold * (
            gold.clamp_min(1e-12).log() - log_probs
        )
    )

    row = {
        "temperature": temperature,
        "KL_loss": kl_loss.item()
    }

    for i, candidate in enumerate(candidate_names):
        row[f"candidate_{candidate}"] = (
            predicted_probs[i].item()
        )

    results.append(row)

    probability_rows.append({
        "temperature": temperature,
        **{
            candidate_names[i]: predicted_probs[i].item()
            for i in range(len(candidate_names))
        }
    })


# --------------------------------------------------
# 5. Display predicted distributions
# --------------------------------------------------

print("\n" + "=" * 70)
print("PREDICTED CANDIDATE PROBABILITIES")
print("=" * 70)

probability_df = pd.DataFrame(probability_rows)

display(
    probability_df.style.format({
        "temperature": "{:.2f}",
        "A": "{:.4f}",
        "B": "{:.4f}",
        "C": "{:.4f}",
        "D": "{:.4f}"
    })
)

# --------------------------------------------------
# 6. Display KL loss comparison
# --------------------------------------------------

print("\n" + "=" * 70)
print("KL LOSS BY TEMPERATURE")
print("=" * 70)

results_df = pd.DataFrame(results)

display(
    results_df[
        ["temperature", "KL_loss"]
    ].style.format({
        "temperature": "{:.2f}",
        "KL_loss": "{:.6f}"
    })
)

# --------------------------------------------------
# 7. Identify best temperature for this example
# --------------------------------------------------

best_index = results_df["KL_loss"].idxmin()

best_temperature = results_df.loc[
    best_index,
    "temperature"
]

best_loss = results_df.loc[
    best_index,
    "KL_loss"
]

print(f"\nLowest KL loss: {best_loss:.6f}")
print(f"Best temperature for this synthetic example: {best_temperature:.2f}")

# --------------------------------------------------
# 8. Final validation
# --------------------------------------------------

for row in probability_rows:

    probability_sum = sum(
        row[candidate]
        for candidate in candidate_names
    )

    assert abs(probability_sum - 1.0) < 1e-6

assert results_df["KL_loss"].notna().all()

print("\nAll temperature sensitivity checks passed.")



# 8. Visualize Temperature Sensitivity

Plot the predicted candidate probabilities at different temperatures against the gold distribution.

Observe how temperature changes the probability assigned to each candidate.


In [ ]:

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import numpy as np

# --------------------------------------------------
# 1. Reuse the synthetic example
# --------------------------------------------------

temperatures = [0.05, 0.10, 0.20, 0.50, 1.00]

candidate_names = ["A", "B", "C", "D"]

gold = torch.tensor([0.50, 0.30, 0.15, 0.05])

z_pred = torch.tensor([1.0, 0.0])

candidate_latents = torch.tensor([
    [1.0, 0.0],
    [0.8, 0.6],
    [0.0, 1.0],
    [-1.0, 0.0]
])

# --------------------------------------------------
# 2. Calculate cosine distances
# --------------------------------------------------

z_pred = F.normalize(z_pred, dim=0)

candidate_latents = F.normalize(
    candidate_latents,
    dim=1
)

distances = 1.0 - torch.matmul(
    candidate_latents,
    z_pred
)

# --------------------------------------------------
# 3. Calculate distributions for each temperature
# --------------------------------------------------

predicted_distributions = []

for temperature in temperatures:

    probabilities = F.softmax(
        -distances / temperature,
        dim=0
    )

    predicted_distributions.append(
        probabilities.numpy()
    )

predicted_distributions = np.array(
    predicted_distributions
)

# --------------------------------------------------
# 4. Plot distributions
# --------------------------------------------------

x = np.arange(len(candidate_names))
width = 0.13

plt.figure(figsize=(11, 6))

# Gold distribution
plt.bar(
    x - 2.5 * width,
    gold.numpy(),
    width,
    label="Gold"
)

# Predicted distributions
for i, temperature in enumerate(temperatures):

    offset = (i - 1.5) * width

    plt.bar(
        x + offset,
        predicted_distributions[i],
        width,
        label=f"Temperature {temperature}"
    )

plt.xticks(x, candidate_names)

plt.ylabel("Probability")
plt.xlabel("Candidate")

plt.title("Gold vs Predicted Candidate Distributions")

plt.ylim(0, 1.05)

plt.legend()
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# --------------------------------------------------
# 5. Plot KL loss against temperature
# --------------------------------------------------

kl_losses = []

for probabilities in predicted_distributions:

    q = torch.tensor(probabilities)

    kl = torch.sum(
        gold * (
            gold.clamp_min(1e-12).log()
            - q.clamp_min(1e-12).log()
        )
    )

    kl_losses.append(kl.item())


plt.figure(figsize=(8, 5))

plt.plot(
    temperatures,
    kl_losses,
    marker="o"
)

plt.xlabel("Temperature")
plt.ylabel("KL Divergence")

plt.title("Temperature vs KL Loss")

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Temperature sensitivity visualization completed.")



# 9. Verify Gradient Flow and Stop-Gradient

JEPA has two encoder branches:

- Online branch: receives gradients and learns through backpropagation.
- Target branch: receives no gradients and is updated through EMA.

In this cell, verify that the candidate target representations are detached while the predicted latent receives gradients from the KL loss.


In [ ]:

import torch

# --------------------------------------------------
# 1. Create synthetic representations
# --------------------------------------------------

torch.manual_seed(42)

latent_dim = 8
num_candidates = 4

# Online branch output: trainable
predicted_latent = torch.randn(
    latent_dim,
    requires_grad=True
)

# Target branch outputs: initially require gradients
# to verify that our loss detaches them.
candidate_target_latents = torch.randn(
    num_candidates,
    latent_dim,
    requires_grad=True
)

gold_probabilities = torch.tensor([
    0.50, 0.30, 0.15, 0.05
])

# --------------------------------------------------
# 2. Calculate JEPA loss
# --------------------------------------------------

result = jepa_candidate_distribution_loss(
    predicted_latent=predicted_latent,
    candidate_target_latents=candidate_target_latents,
    gold_probabilities=gold_probabilities,
    temperature=0.5
)

loss = result[0]  # the loss function returns (loss, probabilities, distances, logits)

# --------------------------------------------------
# 3. Backpropagation
# --------------------------------------------------

loss.backward()

# --------------------------------------------------
# 4. Inspect gradient flow
# --------------------------------------------------

print("=" * 60)
print("JEPA GRADIENT FLOW CHECK")
print("=" * 60)

print(f"Loss: {loss.item():.6f}")

print("\nOnline predicted latent:")
print(f"Requires gradient: {predicted_latent.requires_grad}")
print(f"Gradient exists   : {predicted_latent.grad is not None}")

if predicted_latent.grad is not None:
    print(
        f"Gradient norm     : "
        f"{predicted_latent.grad.norm().item():.6f}"
    )

print("\nTarget candidate latents:")
print(f"Requires gradient: {candidate_target_latents.requires_grad}")
print(f"Gradient exists  : {candidate_target_latents.grad is not None}")

# --------------------------------------------------
# 5. Assert expected behavior
# --------------------------------------------------

assert predicted_latent.grad is not None
assert predicted_latent.grad.norm().item() > 0

assert candidate_target_latents.grad is None

print("\n" + "=" * 60)
print("GRADIENT FLOW VALIDATION PASSED")
print("=" * 60)
print("Online branch : Receives gradients")
print("Target branch : No gradients")



# 10. Exponential Moving Average (EMA) Target Update

Implement the EMA update that synchronizes the target encoder with the online encoder.

The target parameters are updated using the online parameters and an EMA decay of 0.996.

Verify that:
- Both encoders start with identical parameters.
- The online encoder can change independently.
- The target encoder follows through EMA.
- Target parameters do not require gradients.


In [ ]:

import torch
import torch.nn as nn
import copy

# --------------------------------------------------
# 1. Create small synthetic online encoder
# --------------------------------------------------

class ToyEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(8, 16),
            nn.GELU(),
            nn.Linear(16, 8)
        )

    def forward(self, x):
        return self.network(x)


torch.manual_seed(42)

online_encoder = ToyEncoder()

# Target starts as an exact copy
target_encoder = copy.deepcopy(online_encoder)


# --------------------------------------------------
# 2. Configure target encoder
# --------------------------------------------------

EMA_DECAY = CONFIG["target_ema_decay"]

target_encoder.requires_grad_(False)


# --------------------------------------------------
# 3. Define EMA update
# --------------------------------------------------

@torch.no_grad()
def update_ema_target(
    online_model,
    target_model,
    decay=EMA_DECAY
):

    assert 0.0 <= decay < 1.0

    online_parameters = dict(
        online_model.named_parameters()
    )

    target_parameters = dict(
        target_model.named_parameters()
    )

    assert online_parameters.keys() == target_parameters.keys()

    for name, target_param in target_parameters.items():

        online_param = online_parameters[name]

        target_param.mul_(decay).add_(
            online_param,
            alpha=1.0 - decay
        )


# --------------------------------------------------
# 4. Verify initial equality
# --------------------------------------------------

initial_difference = 0.0

for online_param, target_param in zip(
    online_encoder.parameters(),
    target_encoder.parameters()
):

    initial_difference = max(
        initial_difference,
        (online_param - target_param).abs().max().item()
    )

print("Initial maximum parameter difference:", initial_difference)

assert initial_difference == 0.0


# --------------------------------------------------
# 5. Simulate an online encoder update
# --------------------------------------------------

with torch.no_grad():

    for param in online_encoder.parameters():
        param.add_(0.1)


# --------------------------------------------------
# 6. Measure difference before EMA
# --------------------------------------------------

difference_before_ema = max(
    (online_param - target_param).abs().max().item()
    for online_param, target_param in zip(
        online_encoder.parameters(),
        target_encoder.parameters()
    )
)

print("Difference before EMA:", difference_before_ema)


# --------------------------------------------------
# 7. Apply EMA update
# --------------------------------------------------

update_ema_target(
    online_encoder,
    target_encoder,
    decay=EMA_DECAY
)


# --------------------------------------------------
# 8. Measure difference after EMA
# --------------------------------------------------

difference_after_ema = max(
    (online_param - target_param).abs().max().item()
    for online_param, target_param in zip(
        online_encoder.parameters(),
        target_encoder.parameters()
    )
)

print("Difference after EMA:", difference_after_ema)


# --------------------------------------------------
# 9. Validate behavior
# --------------------------------------------------

assert difference_before_ema > 0
assert difference_after_ema < difference_before_ema

assert all(
    not param.requires_grad
    for param in target_encoder.parameters()
)

assert all(
    param.grad is None
    for param in target_encoder.parameters()
)

print("\n" + "=" * 60)
print("EMA VALIDATION PASSED")
print("=" * 60)
print("Online encoder : Independently updated")
print("Target encoder : Updated through EMA")
print(f"EMA decay      : {EMA_DECAY}")


# 11. Load and Inspect ModernBERT

Load the pretrained ModernBERT-base weights and inspect its configuration, layers, and parameter count. We will use these weights to initialize our own online and target encoders; we will not use the library's forward pass for inference.

In [ ]:
import torch
from transformers import AutoConfig, AutoModel

MODEL_NAME = CONFIG["model_name"]

# Load configuration
model_config = AutoConfig.from_pretrained(MODEL_NAME)

# Load pretrained weights
# This is a weight source, not our final inference implementation.
pretrained_model = AutoModel.from_pretrained(
    MODEL_NAME,
    config=model_config,
)

# Inspect model configuration
print("=" * 60)
print("MODERNBERT CONFIGURATION")
print("=" * 60)

print("Model type       :", model_config.model_type)
print("Hidden size      :", model_config.hidden_size)
print("Layers           :", model_config.num_hidden_layers)
print("Attention heads  :", model_config.num_attention_heads)
print("Intermediate size:", model_config.intermediate_size)
print("Max positions    :", model_config.max_position_embeddings)

# Inspect parameter count
total_params = sum(
    parameter.numel()
    for parameter in pretrained_model.parameters()
)

trainable_params = sum(
    parameter.numel()
    for parameter in pretrained_model.parameters()
    if parameter.requires_grad
)

print("\n" + "=" * 60)
print("PARAMETER SUMMARY")
print("=" * 60)

print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size (FP32)   : {total_params * 4 / (1024**3):.2f} GB")

# Inspect top-level architecture
print("\n" + "=" * 60)
print("TOP-LEVEL MODULES")
print("=" * 60)

for name, module in pretrained_model.named_children():
    print(f"{name}: {module.__class__.__name__}")

# Inspect encoder layers
print("\n" + "=" * 60)
print("ENCODER LAYERS")
print("=" * 60)

for name, module in pretrained_model.named_modules():
    if name.count(".") <= 2 and name:
        print(f"{name}: {module.__class__.__name__}")

# Keep the loaded weights available for subsequent cells
print("\nModernBERT weights loaded and ready for inspection.")

# 12. Inspect ModernBERT Internals

Inspect the embedding layer and one transformer layer. Identify the modules we will need to reproduce in our custom forward implementation.

import torch
import torch.nn as nn

print("=" * 60)
print("EMBEDDING MODULE")
print("=" * 60)

print(pretrained_model.embeddings)

print("\n" + "=" * 60)
print("FIRST TRANSFORMER LAYER")
print("=" * 60)

first_layer = pretrained_model.layers[0]
print(first_layer)

print("\n" + "=" * 60)
print("FIRST LAYER: NAMED CHILD MODULES")
print("=" * 60)

for name, module in first_layer.named_children():
    print(f"\n{name}: {module.__class__.__name__}")
    print(module)

print("\n" + "=" * 60)
print("FIRST LAYER: PARAMETER SHAPES")
print("=" * 60)

for name, parameter in first_layer.named_parameters():
    print(f"{name:55s} {tuple(parameter.shape)}")

print("\n" + "=" * 60)
print("FINAL NORMALIZATION")
print("=" * 60)

print(pretrained_model.final_norm)

print("\n" + "=" * 60)
print("ROTARY EMBEDDING")
print("=" * 60)

print(pretrained_model.rotary_emb)

In [ ]:
import torch
import torch.nn as nn

print("=" * 60)
print("EMBEDDING MODULE")
print("=" * 60)

print(pretrained_model.embeddings)

print("\n" + "=" * 60)
print("FIRST TRANSFORMER LAYER")
print("=" * 60)

first_layer = pretrained_model.layers[0]
print(first_layer)

print("\n" + "=" * 60)
print("FIRST LAYER: NAMED CHILD MODULES")
print("=" * 60)

for name, module in first_layer.named_children():
    print(f"\n{name}: {module.__class__.__name__}")
    print(module)

print("\n" + "=" * 60)
print("FIRST LAYER: PARAMETER SHAPES")
print("=" * 60)

for name, parameter in first_layer.named_parameters():
    print(f"{name:55s} {tuple(parameter.shape)}")

print("\n" + "=" * 60)
print("FINAL NORMALIZATION")
print("=" * 60)

print(pretrained_model.final_norm)

print("\n" + "=" * 60)
print("ROTARY EMBEDDING")
print("=" * 60)

print(pretrained_model.rotary_emb)

# 13. Inspect Attention and MLP Implementations

Read the actual forward methods of ModernBERT's attention, MLP, and encoder layer. Focus on QKV splitting, RoPE application, gated MLP computation, and residual connections.

In [ ]:
import inspect

modules_to_inspect = {
    "ModernBertAttention": first_layer.attn,
    "ModernBertMLP": first_layer.mlp,
    "ModernBertEncoderLayer": first_layer,
    "ModernBertRotaryEmbedding": pretrained_model.rotary_emb,
}

for name, module in modules_to_inspect.items():
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    print(inspect.getsource(module.__class__.forward))

In [ ]:
# latent_dim is set in CONFIG (cell 1).
print("latent_dim:", CONFIG["latent_dim"])


# 14. Build the JEPA Online Encoder

Wrap pretrained ModernBERT with a JEPA projection head. ModernBERT produces token representations; masked mean pooling produces a sequence representation, which the projection head maps from 768 to our 256-dimensional JEPA latent space.

In [ ]:
import torch
import torch.nn as nn
import copy

class JEPAEncoder(nn.Module):
    def __init__(self, backbone, hidden_size=768, latent_dim=256):
        super().__init__()

        # Pretrained ModernBERT backbone
        self.backbone = backbone

        # JEPA projection head: 768 -> 768 -> 256
        self.projection = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, latent_dim),
        )

    def forward(self, input_ids, attention_mask):
        # Use the existing ModernBERT forward implementation
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        # Token-level representations: [batch, sequence, hidden]
        token_embeddings = outputs.last_hidden_state

        # Masked mean pooling: exclude padding tokens
        mask = attention_mask.unsqueeze(-1).to(
            token_embeddings.dtype
        )

        summed_embeddings = (token_embeddings * mask).sum(dim=1)
        valid_token_counts = mask.sum(dim=1).clamp(min=1)

        pooled_embeddings = (
            summed_embeddings / valid_token_counts
        )

        # Project into JEPA latent space
        latent = self.projection(pooled_embeddings)

        return latent


# Transfer the already-loaded pretrained ModernBERT into the online encoder
online_encoder = JEPAEncoder(
    backbone=pretrained_model,
    hidden_size=model_config.hidden_size,
    latent_dim=CONFIG["latent_dim"],
)

online_encoder = online_encoder.to(CONFIG["device"])
online_encoder.train()


# Test with synthetic token inputs
batch_size = 2
sequence_length = 16

test_input_ids = torch.randint(
    low=0,
    high=model_config.vocab_size,
    size=(batch_size, sequence_length),
    device=CONFIG["device"],
)

test_attention_mask = torch.ones(
    batch_size,
    sequence_length,
    dtype=torch.long,
    device=CONFIG["device"],
)

# Forward pass
test_latent = online_encoder(
    test_input_ids,
    test_attention_mask,
)

# Verify gradient flow through both backbone and projection head
test_loss = test_latent.pow(2).mean()
test_loss.backward()

backbone_gradients = any(
    p.grad is not None and p.grad.abs().sum() > 0
    for p in online_encoder.backbone.parameters()
)

projection_gradients = any(
    p.grad is not None and p.grad.abs().sum() > 0
    for p in online_encoder.projection.parameters()
)

print("=" * 60)
print("JEPA ONLINE ENCODER")
print("=" * 60)

print("Input shape       :", tuple(test_input_ids.shape))
print("Token output shape:", (batch_size, sequence_length, model_config.hidden_size))
print("Latent shape      :", tuple(test_latent.shape))
print("Latent dimension  :", CONFIG["latent_dim"])

print("\nGradient validation")
print("Backbone gradients  :", backbone_gradients)
print("Projection gradients:", projection_gradients)

assert test_latent.shape == (batch_size, CONFIG["latent_dim"])
assert backbone_gradients
assert projection_gradients

print("\nONLINE ENCODER VALIDATION PASSED")

# Clear test gradients before actual training
online_encoder.zero_grad(set_to_none=True)

# 15. Create the EMA Target Encoder

Create a frozen copy of the online JEPA encoder. Initialize it with identical weights. During training, update its parameters using EMA rather than backpropagation.

In [ ]:
import copy
import torch

# Deep copy the complete online encoder
target_encoder = copy.deepcopy(online_encoder)

# Target encoder must not receive gradients
target_encoder.requires_grad_(False)
target_encoder.eval()

# EMA update function
@torch.no_grad()
def update_ema_target(online_encoder, target_encoder, decay=CONFIG["target_ema_decay"]):

    online_params = dict(online_encoder.named_parameters())
    target_params = dict(target_encoder.named_parameters())

    assert online_params.keys() == target_params.keys()

    for name in online_params:
        target_params[name].mul_(decay).add_(
            online_params[name],
            alpha=1.0 - decay
        )

    # Keep buffers synchronized (if the architecture has any)
    online_buffers = dict(online_encoder.named_buffers())
    target_buffers = dict(target_encoder.named_buffers())

    for name in online_buffers:
        if name in target_buffers:
            target_buffers[name].copy_(online_buffers[name])


# Validate initial equality
max_initial_difference = max(
    (p_online - p_target).abs().max().item()
    for p_online, p_target in zip(
        online_encoder.parameters(),
        target_encoder.parameters()
    )
)

# Change online parameters to simulate an optimizer update
with torch.no_grad():
    for parameter in online_encoder.parameters():
        parameter.add_(0.01)

# Verify target has not changed yet
difference_before_ema = max(
    (p_online - p_target).abs().max().item()
    for p_online, p_target in zip(
        online_encoder.parameters(),
        target_encoder.parameters()
    )
)

# Apply EMA
EMA_DECAY = CONFIG["target_ema_decay"]

update_ema_target(
    online_encoder,
    target_encoder,
    decay=EMA_DECAY
)

# Verify target follows online encoder
difference_after_ema = max(
    (p_online - p_target).abs().max().item()
    for p_online, p_target in zip(
        online_encoder.parameters(),
        target_encoder.parameters()
    )
)

target_has_gradients = any(
    p.requires_grad or p.grad is not None
    for p in target_encoder.parameters()
)

print("=" * 60)
print("JEPA TARGET ENCODER VALIDATION")
print("=" * 60)

print("Initial parameter difference:", max_initial_difference)
print("Difference before EMA       :", difference_before_ema)
print("Difference after EMA        :", difference_after_ema)
print("EMA decay                   :", EMA_DECAY)
print("Target has gradients        :", target_has_gradients)

assert max_initial_difference == 0.0
assert difference_after_ema < difference_before_ema
assert not target_has_gradients

print("\nTARGET ENCODER VALIDATION PASSED")

# 16. Build the JEPA Predictor

The predictor maps the online encoder's 256-dimensional context latent into the same latent space as the target encoder's candidate representations. It learns to predict the target representation from the context.

In [ ]:
import torch
import torch.nn as nn

class JEPAPredictor(nn.Module):
    def __init__(self, latent_dim=CONFIG["latent_dim"], hidden_dim=CONFIG["predictor_hidden_dim"]):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, context_latent):
        return self.network(context_latent)


# Create predictor
predictor = JEPAPredictor(
    latent_dim=CONFIG["latent_dim"],
    hidden_dim=CONFIG["predictor_hidden_dim"],
).to(CONFIG["device"])

# Test predictor
batch_size = 4

test_context_latent = torch.randn(
    batch_size,
    CONFIG["latent_dim"],
    device=CONFIG["device"],
)

predicted_latent = predictor(test_context_latent)

# Verify shape and gradient flow
test_loss = predicted_latent.pow(2).mean()
test_loss.backward()

predictor_gradients = any(
    p.grad is not None and p.grad.abs().sum() > 0
    for p in predictor.parameters()
)

print("=" * 60)
print("JEPA PREDICTOR VALIDATION")
print("=" * 60)

print("Context latent shape :", tuple(test_context_latent.shape))
print("Predicted latent shape:", tuple(predicted_latent.shape))
print("Predictor parameters :", sum(
    p.numel() for p in predictor.parameters()
))
print("Gradient flow        :", predictor_gradients)

assert predicted_latent.shape == test_context_latent.shape
assert predictor_gradients

print("\nPREDICTOR VALIDATION PASSED")

predictor.zero_grad(set_to_none=True)

# 17. Test the Complete JEPA Forward Path

Pass a context through the online encoder and predictor. Separately encode candidate inputs using the frozen target encoder. Verify that both branches produce compatible 256-dimensional latents and that gradients flow only through the online branch and predictor.

In [ ]:
import torch

# Put online encoder and predictor in training mode
online_encoder.train()
predictor.train()

# Target encoder remains frozen
target_encoder.eval()

batch_size = 2
sequence_length = 16
num_candidates = 4

# Synthetic context input
context_input_ids = torch.randint(
    0,
    model_config.vocab_size,
    (batch_size, sequence_length),
    device=CONFIG["device"],
)

context_attention_mask = torch.ones(
    batch_size,
    sequence_length,
    dtype=torch.long,
    device=CONFIG["device"],
)

# Synthetic candidate inputs:
# [batch, candidates, sequence]
candidate_input_ids = torch.randint(
    0,
    model_config.vocab_size,
    (batch_size, num_candidates, sequence_length),
    device=CONFIG["device"],
)

candidate_attention_mask = torch.ones(
    batch_size,
    num_candidates,
    sequence_length,
    dtype=torch.long,
    device=CONFIG["device"],
)

# 1. Online branch: context -> predicted latent
context_latent = online_encoder(
    context_input_ids,
    context_attention_mask,
)

predicted_latent = predictor(context_latent)

# 2. Target branch: encode each candidate independently
flat_candidate_ids = candidate_input_ids.reshape(
    batch_size * num_candidates,
    sequence_length,
)

flat_candidate_mask = candidate_attention_mask.reshape(
    batch_size * num_candidates,
    sequence_length,
)

with torch.no_grad():
    flat_candidate_latents = target_encoder(
        flat_candidate_ids,
        flat_candidate_mask,
    )

candidate_latents = flat_candidate_latents.reshape(
    batch_size,
    num_candidates,
    CONFIG["latent_dim"],
)

# 3. Verify gradient flow
# Use a simple synthetic objective for this structural test.
test_loss = predicted_latent.pow(2).mean()
test_loss.backward()

online_has_gradients = any(
    p.grad is not None and p.grad.abs().sum() > 0
    for p in online_encoder.parameters()
)

predictor_has_gradients = any(
    p.grad is not None and p.grad.abs().sum() > 0
    for p in predictor.parameters()
)

target_has_gradients = any(
    p.grad is not None
    for p in target_encoder.parameters()
)

print("=" * 60)
print("COMPLETE JEPA FORWARD PATH")
print("=" * 60)

print("Context latent shape   :", tuple(context_latent.shape))
print("Predicted latent shape :", tuple(predicted_latent.shape))
print("Candidate latent shape:", tuple(candidate_latents.shape))

print("\nGradient validation")
print("Online encoder :", online_has_gradients)
print("Predictor      :", predictor_has_gradients)
print("Target encoder :", target_has_gradients)

assert predicted_latent.shape == (
    batch_size,
    CONFIG["latent_dim"],
)

assert candidate_latents.shape == (
    batch_size,
    num_candidates,
    CONFIG["latent_dim"],
)

assert online_has_gradients
assert predictor_has_gradients
assert not target_has_gradients

print("\nCOMPLETE JEPA FORWARD PATH PASSED")

# Clear gradients before the next experiment
online_encoder.zero_grad(set_to_none=True)
predictor.zero_grad(set_to_none=True)

# 18. Connect the JEPA Candidate Distribution Loss

Use the predicted context latent and candidate target latents to calculate a predicted probability distribution over candidates. Compare it with the dataset's gold probability distribution using KL divergence.

In [ ]:
import torch

# Example gold distributions for two samples.
# Each row corresponds to the candidate order in candidate_latents.
gold_probabilities = torch.tensor(
    [
        [0.50, 0.30, 0.15, 0.05],
        [0.10, 0.20, 0.50, 0.20],
    ],
    dtype=torch.float32,
    device=CONFIG["device"],
)

# Calculate JEPA candidate distribution loss
loss, predicted_probabilities, distances, logits = (
    jepa_candidate_distribution_loss(
        predicted_latent=predicted_latent,
        candidate_target_latents=candidate_latents,
        gold_probabilities=gold_probabilities,
        temperature=CONFIG["temperature"],
    )
)

print("=" * 60)
print("JEPA CANDIDATE DISTRIBUTION LOSS")
print("=" * 60)

print("Predicted latent shape :", tuple(predicted_latent.shape))
print("Candidate latent shape :", tuple(candidate_latents.shape))
print("Gold distribution shape:", tuple(gold_probabilities.shape))

print("\nGold distributions:")
print(gold_probabilities.detach().cpu())

print("\nPredicted distributions:")
print(predicted_probabilities.detach().cpu())

print("\nCandidate distances:")
print(distances.detach().cpu())

print("\nKL divergence loss:", loss.item())

# Verify probability distributions
assert torch.allclose(
    predicted_probabilities.sum(dim=-1),
    torch.ones(batch_size, device=CONFIG["device"]),
    atol=1e-5,
)

assert torch.isfinite(loss)
assert loss.item() >= 0

print("\nDISTRIBUTION LOSS VALIDATION PASSED")

# 19. Verify End-to-End JEPA Gradient Flow

Calculate the actual candidate distribution loss and backpropagate it. Verify that gradients reach the online ModernBERT backbone, projection head, and predictor, while the EMA target encoder remains frozen.

In [ ]:

# Clear previous gradients
online_encoder.zero_grad(set_to_none=True)
predictor.zero_grad(set_to_none=True)
target_encoder.zero_grad(set_to_none=True)

# Recompute online branch
context_latent = online_encoder(
    context_input_ids,
    context_attention_mask,
)

predicted_latent = predictor(context_latent)

# Recompute target branch
with torch.no_grad():
    flat_candidate_latents = target_encoder(
        flat_candidate_ids,
        flat_candidate_mask,
    )

candidate_latents = flat_candidate_latents.reshape(
    batch_size,
    num_candidates,
    CONFIG["latent_dim"],
)

# Actual JEPA loss
loss, predicted_probabilities, distances, logits = (
    jepa_candidate_distribution_loss(
        predicted_latent,
        candidate_latents,
        gold_probabilities,
        temperature=CONFIG["temperature"],
    )
)

# Backpropagation
loss.backward()

def gradient_summary(module):
    gradients = [
        p.grad for p in module.parameters()
        if p.grad is not None
    ]

    return {
        "gradient_tensors": len(gradients),
        "gradient_norm": (
            torch.sqrt(sum(g.detach().pow(2).sum() for g in gradients)).item()
            if gradients else 0.0
        ),
    }

backbone_grad = gradient_summary(online_encoder.backbone)
projection_grad = gradient_summary(online_encoder.projection)
predictor_grad = gradient_summary(predictor)

target_gradients = sum(
    p.grad is not None
    for p in target_encoder.parameters()
)

print("=" * 60)
print("END-TO-END JEPA GRADIENT FLOW")
print("=" * 60)

print(f"Loss: {loss.item():.6f}")

print("\nOnline ModernBERT backbone:", backbone_grad)
print("JEPA projection head      :", projection_grad)
print("JEPA predictor            :", predictor_grad)
print("Target encoder gradients  :", target_gradients)

assert backbone_grad["gradient_tensors"] > 0
assert backbone_grad["gradient_norm"] > 0
assert projection_grad["gradient_norm"] > 0
assert predictor_grad["gradient_norm"] > 0
assert target_gradients == 0

print("\nEND-TO-END GRADIENT VALIDATION PASSED")

# Clear gradients before training setup
online_encoder.zero_grad(set_to_none=True)
predictor.zero_grad(set_to_none=True)


# 20. Configure the JEPA Training Optimizer

Create an AdamW optimizer for the online ModernBERT encoder, projection head, and predictor. Exclude the frozen target encoder. Use separate learning rates for the pretrained backbone and newly initialized JEPA components.

In [ ]:

import torch

# Separate learning rates:
# Lower rate for pretrained ModernBERT
# Higher rate for newly initialized JEPA components

BACKBONE_LR = CONFIG["backbone_lr"]
JEPA_LR = CONFIG["jepa_lr"]
WEIGHT_DECAY = CONFIG["weight_decay"]

optimizer = torch.optim.AdamW(
    [
        {
            "params": online_encoder.backbone.parameters(),
            "lr": BACKBONE_LR,
            "name": "modernbert_backbone",
        },
        {
            "params": online_encoder.projection.parameters(),
            "lr": JEPA_LR,
            "name": "jepa_projection",
        },
        {
            "params": predictor.parameters(),
            "lr": JEPA_LR,
            "name": "jepa_predictor",
        },
    ],
    weight_decay=WEIGHT_DECAY,
)

# Verify target encoder is excluded
target_parameter_ids = {
    id(p) for p in target_encoder.parameters()
}

optimizer_parameter_ids = {
    id(p)
    for group in optimizer.param_groups
    for p in group["params"]
}

assert target_parameter_ids.isdisjoint(optimizer_parameter_ids)

# Verify optimizer configuration
print("=" * 60)
print("JEPA OPTIMIZER CONFIGURATION")
print("=" * 60)

for group in optimizer.param_groups:
    parameter_count = sum(
        p.numel() for p in group["params"]
    )

    print(f"\nGroup: {group['name']}")
    print(f"Learning rate: {group['lr']}")
    print(f"Parameters   : {parameter_count:,}")

print("\nTotal optimized parameters:",
      f"{sum(p.numel() for g in optimizer.param_groups for p in g['params']):,}")

print("Target encoder excluded:", True)

print("\nOPTIMIZER VALIDATION PASSED")


# 21. Restore the Pretrained Starting Point

Reload the original pretrained ModernBERT backbone, recreate the online and EMA target encoders, and initialize the predictor and optimizer again. This removes the artificial parameter changes from our EMA validation and gives training a clean starting point.

In [ ]:

import copy
import torch
from transformers import AutoModel

def build_clean_jepa():
    """Fresh pretrained ModernBERT -> online encoder, EMA target, predictor, optimizer."""
    # Reload original pretrained ModernBERT weights
    clean_backbone = AutoModel.from_pretrained(
        MODEL_NAME,
        config=model_config,
    )

    clean_backbone = clean_backbone.to(CONFIG["device"])

    # Recreate online JEPA encoder
    online_encoder = JEPAEncoder(
        backbone=clean_backbone,
        hidden_size=model_config.hidden_size,
        latent_dim=CONFIG["latent_dim"],
    ).to(CONFIG["device"])

    # Recreate target encoder from identical online weights
    target_encoder = copy.deepcopy(online_encoder)
    target_encoder.requires_grad_(False)
    target_encoder.eval()

    # Reinitialize predictor
    predictor = JEPAPredictor(
        latent_dim=CONFIG["latent_dim"],
        hidden_dim=CONFIG["predictor_hidden_dim"],
    ).to(CONFIG["device"])

    # Rebuild optimizer
    optimizer = torch.optim.AdamW(
        [
            {
                "params": online_encoder.backbone.parameters(),
                "lr": CONFIG["backbone_lr"],
            },
            {
                "params": online_encoder.projection.parameters(),
                "lr": CONFIG["jepa_lr"],
            },
            {
                "params": predictor.parameters(),
                "lr": CONFIG["jepa_lr"],
            },
        ],
        weight_decay=CONFIG["weight_decay"],
    )

    return online_encoder, target_encoder, predictor, optimizer


online_encoder, target_encoder, predictor, optimizer = build_clean_jepa()

# Verify online and target start identically
max_difference = max(
    (p_online - p_target).abs().max().item()
    for p_online, p_target in zip(
        online_encoder.parameters(),
        target_encoder.parameters(),
    )
)

print("=" * 60)
print("CLEAN JEPA INITIALIZATION")
print("=" * 60)

print("Backbone parameters:",
      sum(p.numel() for p in online_encoder.backbone.parameters()))

print("Online parameters:",
      sum(p.numel() for p in online_encoder.parameters()))

print("Predictor parameters:",
      sum(p.numel() for p in predictor.parameters()))

print("Initial online-target difference:", max_difference)
print("Target gradients enabled:",
      any(p.requires_grad for p in target_encoder.parameters()))

assert max_difference == 0.0
assert not any(p.requires_grad for p in target_encoder.parameters())

print("\nCLEAN INITIALIZATION PASSED")


# 22. Prepare JEPA Training Samples

Convert the validated training split into JEPA samples. Each sample contains the state, question instructions, candidate descriptions, and gold probability distribution. We retain variable candidate counts and use the existing tokenizer with a 1024-token maximum sequence length.

In [ ]:

from torch.utils.data import Dataset

MAX_LENGTH = CONFIG["max_sequence_length"]

class JEPADataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=CONFIG["max_sequence_length"]):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def build_online_text(self, sample):
        return (
            "State:\n"
            + str(sample["state"])
            + "\n\nQuestion:\n"
            + str(sample["instructions"])
            + "\n\nCandidates:\n"
            + "\n".join(
                f'{candidate["key"]}: {candidate["description"]}'
                for candidate in sample["candidates"]
            )
        )

    def build_target_text(self, sample, candidate):
        return (
            "State:\n"
            + str(sample["state"])
            + "\n\nQuestion:\n"
            + str(sample["instructions"])
            + "\n\nSelected candidate:\n"
            + str(candidate["description"])
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]

        online_text = self.build_online_text(sample)

        online_tokens = self.tokenizer(
            online_text,
            truncation=CONFIG["truncation"],
            max_length=self.max_length,
            padding=False,
            return_tensors="pt",
        )

        candidate_tokens = []

        for candidate in sample["candidates"]:
            text = self.build_target_text(sample, candidate)

            tokens = self.tokenizer(
                text,
                truncation=CONFIG["truncation"],
                max_length=self.max_length,
                padding=False,
                return_tensors="pt",
            )

            candidate_tokens.append({
                "input_ids": tokens["input_ids"].squeeze(0),
                "attention_mask": tokens["attention_mask"].squeeze(0),
            })

        gold = torch.tensor(
            [
                sample["gold_probabilities"][candidate["key"]]
                for candidate in sample["candidates"]
            ],
            dtype=torch.float32,
        )

        return {
            "record_id": sample["record_id"],
            "question_id": sample["question_id"],
            "question_type": sample["question_type"],
            "candidate_keys": [
                candidate["key"]
                for candidate in sample["candidates"]
            ],
            "online": {
                "input_ids": online_tokens["input_ids"].squeeze(0),
                "attention_mask": online_tokens["attention_mask"].squeeze(0),
            },
            "candidates": candidate_tokens,
            "gold_probabilities": gold,
        }


# Use the group-aware splits from Cell 5
train_dataset = JEPADataset(
    train_samples,
    tokenizer,
    MAX_LENGTH,
)

validation_dataset = JEPADataset(
    validation_samples,
    tokenizer,
    MAX_LENGTH,
)

test_dataset = JEPADataset(
    test_samples,
    tokenizer,
    MAX_LENGTH,
)

print("=" * 60)
print("JEPA DATASET PREPARATION")
print("=" * 60)

print("Training samples  :", len(train_dataset))
print("Validation samples:", len(validation_dataset))
print("Test samples      :", len(test_dataset))

# Inspect one sample
example = train_dataset[0]

print("\nExample question type:", example["question_type"])
print("Example question ID  :", example["question_id"])
print("Online token count   :", len(example["online"]["input_ids"]))
print("Candidate count      :", len(example["candidates"]))
print("Candidate token counts:",
      [len(c["input_ids"]) for c in example["candidates"]])
print("Gold distribution    :", example["gold_probabilities"].tolist())

assert len(train_dataset) > 0
assert len(validation_dataset) > 0
assert len(test_dataset) > 0

print("\nDATASET PREPARATION PASSED")


# 23. Build the JEPA Batch Collator

Create batches containing context inputs, candidate inputs, candidate masks, and gold distributions. Support variable sequence lengths and variable candidate counts by padding within each batch. Padding candidates must be masked out of the loss.

In [ ]:

import torch
from torch.utils.data import DataLoader

PAD_TOKEN_ID = tokenizer.pad_token_id

def jepa_collate_fn(batch):
    batch_size = len(batch)
    max_candidates = max(len(item["candidates"]) for item in batch)

    # Context inputs
    context_ids = [item["online"]["input_ids"] for item in batch]
    context_masks = [item["online"]["attention_mask"] for item in batch]

    context_input_ids = torch.nn.utils.rnn.pad_sequence(
        context_ids,
        batch_first=True,
        padding_value=PAD_TOKEN_ID,
    )

    context_attention_mask = torch.nn.utils.rnn.pad_sequence(
        context_masks,
        batch_first=True,
        padding_value=0,
    )

    # Candidate inputs: flatten B x N, then pad sequence dimension
    flat_candidate_ids = []
    flat_candidate_masks = []

    candidate_mask = torch.zeros(
        batch_size, max_candidates, dtype=torch.bool
    )

    gold_probabilities = torch.zeros(
        batch_size, max_candidates, dtype=torch.float32
    )

    candidate_keys = []

    for batch_idx, item in enumerate(batch):
        keys = item["candidate_keys"]
        candidate_keys.append(keys)

        gold_probabilities[batch_idx, :len(keys)] = (
            item["gold_probabilities"]
        )

        for candidate_idx, candidate in enumerate(item["candidates"]):
            flat_candidate_ids.append(candidate["input_ids"])
            flat_candidate_masks.append(candidate["attention_mask"])
            candidate_mask[batch_idx, candidate_idx] = True

    candidate_input_ids = torch.nn.utils.rnn.pad_sequence(
        flat_candidate_ids,
        batch_first=True,
        padding_value=PAD_TOKEN_ID,
    )

    candidate_attention_mask = torch.nn.utils.rnn.pad_sequence(
        flat_candidate_masks,
        batch_first=True,
        padding_value=0,
    )

    return {
        "record_ids": [item["record_id"] for item in batch],
        "question_ids": [item["question_id"] for item in batch],
        "question_types": [item["question_type"] for item in batch],
        "candidate_keys": candidate_keys,
        "context_input_ids": context_input_ids,
        "context_attention_mask": context_attention_mask,
        "candidate_input_ids": candidate_input_ids,
        "candidate_attention_mask": candidate_attention_mask,
        "candidate_mask": candidate_mask,
        "gold_probabilities": gold_probabilities,
    }


# Test with mixed NOUL and Choice examples
test_batch = [
    train_dataset[i] for i in range(8)
]

collated = jepa_collate_fn(test_batch)

print("=" * 60)
print("JEPA BATCH COLLATOR")
print("=" * 60)

for key, value in collated.items():
    if isinstance(value, torch.Tensor):
        print(f"{key:28s}: {tuple(value.shape)}")

print("\nCandidate counts:",
      collated["candidate_mask"].sum(dim=1).tolist())

print("Gold probability sums:",
      collated["gold_probabilities"].sum(dim=1).tolist())

assert torch.all(
    collated["candidate_mask"].sum(dim=1) >= 2
)

assert torch.allclose(
    collated["gold_probabilities"].sum(dim=1),
    torch.ones(len(test_batch)),
    atol=0.01,
)

print("\nBATCH COLLATOR VALIDATION PASSED")


# 24. Make Candidate Distribution Loss Padding-Aware

Extend the JEPA loss to accept a candidate mask. Invalid padded candidate positions must receive zero probability and contribute nothing to the KL divergence. Preserve support for single samples and batches.

In [ ]:

import torch
import torch.nn.functional as F

def jepa_candidate_distribution_loss(
    predicted_latent,
    candidate_target_latents,
    gold_probabilities,
    temperature=CONFIG["temperature"],
    candidate_mask=None,
):
    """
    Supports single samples and batches with variable candidate counts.

    Single:
        predicted:  [D]
        candidates: [N, D]
        gold:       [N]

    Batch:
        predicted:  [B, D]
        candidates: [B, N, D]
        gold:       [B, N]
        mask:       [B, N]
    """

    single_sample = predicted_latent.ndim == 1

    if single_sample:
        predicted_latent = predicted_latent.unsqueeze(0)
        candidate_target_latents = candidate_target_latents.unsqueeze(0)
        gold_probabilities = gold_probabilities.unsqueeze(0)

        if candidate_mask is not None:
            candidate_mask = candidate_mask.unsqueeze(0)

    assert predicted_latent.ndim == 2
    assert candidate_target_latents.ndim == 3
    assert gold_probabilities.ndim == 2
    assert temperature > 0

    batch_size, num_candidates, latent_dim = candidate_target_latents.shape

    assert predicted_latent.shape == (batch_size, latent_dim)
    assert gold_probabilities.shape == (batch_size, num_candidates)

    if candidate_mask is None:
        candidate_mask = torch.ones(
            batch_size,
            num_candidates,
            dtype=torch.bool,
            device=predicted_latent.device,
        )
    else:
        candidate_mask = candidate_mask.to(
            device=predicted_latent.device,
            dtype=torch.bool,
        )

    assert candidate_mask.shape == (batch_size, num_candidates)
    assert candidate_mask.any(dim=-1).all()

    assert torch.isfinite(predicted_latent).all()
    assert torch.isfinite(candidate_target_latents).all()
    assert torch.isfinite(gold_probabilities).all()
    assert (gold_probabilities >= 0).all()

    # Padded candidates must have zero gold probability.
    assert torch.all(
        gold_probabilities.masked_select(~candidate_mask) == 0
    )

    gold_sums = gold_probabilities.sum(dim=-1)
    assert torch.allclose(
        gold_sums,
        torch.ones_like(gold_sums),
        atol=0.01,
    )

    predicted_norm = F.normalize(predicted_latent, dim=-1)
    candidate_norm = F.normalize(
        candidate_target_latents.detach(),
        dim=-1,
    )

    distances = 1.0 - torch.einsum(
        "bd,bnd->bn",
        predicted_norm,
        candidate_norm,
    )

    logits = -distances / temperature

    # Exclude padded candidates from softmax.
    logits = logits.masked_fill(
        ~candidate_mask,
        torch.finfo(logits.dtype).min,
    )

    log_predicted_probs = F.log_softmax(logits, dim=-1)
    predicted_probs = log_predicted_probs.exp()

    safe_gold = gold_probabilities.clamp_min(1e-12)

    kl_terms = torch.where(
        gold_probabilities > 0,
        gold_probabilities * (
            safe_gold.log() - log_predicted_probs
        ),
        torch.zeros_like(gold_probabilities),
    )

    loss = kl_terms.sum(dim=-1).mean()

    if single_sample:
        return (
            loss,
            predicted_probs.squeeze(0),
            distances.squeeze(0),
            logits.squeeze(0),
        )

    return loss, predicted_probs, distances, logits


# Validate using the collated batch from Cell 23.
# Synthetic latents here isolate the loss behavior.
B, N = collated["candidate_mask"].shape
D = CONFIG["latent_dim"]

test_predicted = torch.randn(
    B, D, device=CONFIG["device"]
)

test_candidates = torch.randn(
    B, N, D, device=CONFIG["device"]
)

test_gold = collated["gold_probabilities"].to(CONFIG["device"])
test_mask = collated["candidate_mask"].to(CONFIG["device"])

loss, probs, distances, logits = jepa_candidate_distribution_loss(
    test_predicted,
    test_candidates,
    test_gold,
    temperature=CONFIG["temperature"],
    candidate_mask=test_mask,
)

print("=" * 60)
print("PADDING-AWARE JEPA LOSS")
print("=" * 60)

print("Candidate mask shape:", tuple(test_mask.shape))
print("Probability shape   :", tuple(probs.shape))
print("Loss                :", loss.item())

print("\nCandidate counts:")
print(test_mask.sum(dim=-1).tolist())

print("\nPredicted probabilities:")
print(probs.detach().cpu())

print("\nProbability sums:")
print(probs.sum(dim=-1).detach().cpu())

# Check padded positions receive exactly zero probability.
assert torch.all(probs[~test_mask] == 0)

# Each valid candidate distribution must sum to one.
assert torch.allclose(
    probs.sum(dim=-1),
    torch.ones(B, device=CONFIG["device"]),
    atol=1e-6,
)

assert torch.isfinite(loss)

print("\nPADDING-AWARE LOSS VALIDATION PASSED")


# 25. Create Training DataLoaders

Create shuffled training batches and deterministic validation/test batches. Use the custom collator to support variable sequence lengths and candidate counts. Start with a small batch size to validate memory usage on the GPU.

In [ ]:

from torch.utils.data import DataLoader

BATCH_SIZE = CONFIG["batch_size"]
NUM_WORKERS = CONFIG["num_workers"]

# The training order is drawn from this generator. Section 29 reseeds it per epoch,
# so a resumed run sees the same batches in the same order.
train_generator = torch.Generator()
train_generator.manual_seed(CONFIG["seed"])

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=NUM_WORKERS,
    collate_fn=jepa_collate_fn,
    pin_memory=torch.cuda.is_available(),
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=jepa_collate_fn,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=jepa_collate_fn,
    pin_memory=torch.cuda.is_available(),
)

# Inspect one training batch
batch = next(iter(train_loader))

print("=" * 60)
print("JEPA DATALOADER VALIDATION")
print("=" * 60)

print("Training batches  :", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Test batches      :", len(test_loader))

print("\nFirst training batch:")
for key, value in batch.items():
    if isinstance(value, torch.Tensor):
        print(f"{key:28s}: {tuple(value.shape)}")

print("\nCandidate counts:",
      batch["candidate_mask"].sum(dim=1).tolist())

print("Question types:", batch["question_types"])

# Check gold distributions
assert torch.allclose(
    batch["gold_probabilities"].sum(dim=-1),
    torch.ones(BATCH_SIZE),
    atol=0.01,
)

print("\nDATALOADER VALIDATION PASSED")


# 25b. Answer Targets

Each candidate is compared with a **target latent**. `target_mode` chooses it:

- **`encoder`** (this run): ModernBERT's own encoding of the answer word, `yes` for the true candidate and `no` for the false one. No state and no question are in it.
  - **How:** the word is repeated `target_repeats` times ("yes yes yes …"), averaged over the word tokens (`[CLS]` and `[SEP]` left out), and passed through the projection head at its starting weights.
  - **Why repeat:** a single word shares `[CLS]`/`[SEP]` with the other word, so `yes` and `no` come out 0.86–0.88 alike. Repeated 64 times they're 0.54 alike (`decoder_jepa_target_check.ipynb`).
  - **Fixed:** the two vectors are computed once and never change, so no target encoder or EMA is needed during training.
- **`fixed`:** two opposite unit vectors. The same model and loss without ModernBERT's meaning of yes/no; the comparison for `encoder`.
- **`candidate_text`:** the first design, the EMA target encoder on state + question + candidate.

**The gate.** With cosine `c` between the two targets, even a perfect prediction gives the gold at most `sigmoid((1 − c) / τ)`. The targets must let a gold of `target_gate_gold` be reached.

In [ ]:
import math

import torch
import torch.nn.functional as F


def answer_text(word):
    return CONFIG["target_separator"].join([word] * CONFIG["target_repeats"])


@torch.no_grad()
def encode_answer(encoder, word):
    """The encoder's latent of the repeated answer word, pooled as target_pooling says."""
    tokens = tokenizer(answer_text(word), return_tensors="pt", return_special_tokens_mask=True)
    special = tokens.pop("special_tokens_mask")[0].bool()
    tokens = {k: v.to(CONFIG["device"]) for k, v in tokens.items()}

    was_training = encoder.training
    encoder.eval()

    states = encoder.backbone(**tokens).last_hidden_state[0]      # [tokens, hidden]
    special = special.to(states.device)

    if CONFIG["target_pooling"] == "mean_words":
        pooled = states[~special].mean(0)
    elif CONFIG["target_pooling"] == "mean_all":
        pooled = states.mean(0)
    else:
        raise ValueError(CONFIG["target_pooling"])

    latent = encoder.projection(pooled.unsqueeze(0))[0]
    encoder.train(was_training)

    return latent.float().detach().clone()


def build_answer_targets(encoder):
    """{candidate key: fixed target latent}; None for the first design (candidate_text)."""
    mode = CONFIG["target_mode"]

    if mode == "candidate_text":
        return None

    if mode == "encoder":
        positive, negative = CONFIG["target_words"]
        t_true, t_false = encode_answer(encoder, positive), encode_answer(encoder, negative)

    elif mode == "fixed":
        generator = torch.Generator().manual_seed(CONFIG["seed"])
        u = F.normalize(torch.randn(CONFIG["latent_dim"], generator=generator), dim=0)
        t_true, t_false = u, -u

    else:
        raise ValueError(mode)

    return {"true": t_true.to(CONFIG["device"]), "false": t_false.to(CONFIG["device"])}


def report_answer_targets(targets):
    if targets is None:
        print("target_mode = candidate_text: candidates are encoded by the EMA target encoder")
        return

    c = F.cosine_similarity(targets["true"], targets["false"], dim=0).item()
    tau = CONFIG["temperature"]
    p_max = 1 / (1 + math.exp(-(1 - c) / tau))
    gate_gold = CONFIG["target_gate_gold"]
    gate = 1 - tau * math.log(gate_gold / (1 - gate_gold))

    print("=" * 60)
    print("ANSWER TARGETS")
    print("=" * 60)
    print(f"Mode               : {CONFIG['target_mode']}")
    if CONFIG["target_mode"] == "encoder":
        print(f"Words              : {CONFIG['target_words']} x {CONFIG['target_repeats']}, "
              f"pooling {CONFIG['target_pooling']}")
    print(f"cos(true, false)   : {c:.4f}")
    print(f"Highest gold prob. : {p_max:.4f} (tau {tau})")
    print(f"Gate               : cos <= {gate:.3f} (gold {gate_gold})")

    assert c <= gate, "The answer targets are too alike: use other words, more repeats, or target_mode='fixed'."
    print("\nANSWER TARGETS PASS THE GATE")


def candidate_latents_for(batch):
    """The candidates' target latents, [batch, candidates, latent_dim]; padded slots stay zero."""
    mask = batch["candidate_mask"]
    device_ = CONFIG["device"]

    if CONFIG["target_mode"] == "candidate_text":
        with torch.no_grad():
            flat = target_encoder(
                batch["candidate_input_ids"].to(device_),
                batch["candidate_attention_mask"].to(device_),
            )
        latents = torch.zeros(mask.shape[0], mask.shape[1], flat.shape[-1], dtype=flat.dtype, device=device_)
        offset = 0
        for i in range(mask.shape[0]):
            count = int(mask[i].sum().item())
            latents[i, :count] = flat[offset:offset + count]
            offset += count
        assert offset == flat.shape[0]
        return latents

    dim = next(iter(ANSWER_TARGETS.values())).shape[-1]
    latents = torch.zeros(mask.shape[0], mask.shape[1], dim, device=device_)
    for i, keys in enumerate(batch["candidate_keys"]):
        for j, key in enumerate(keys):
            latents[i, j] = ANSWER_TARGETS[key]
    return latents


# Targets for the demonstration cells below; section 29 builds them again from its own fresh start.
ANSWER_TARGETS = build_answer_targets(online_encoder)
report_answer_targets(ANSWER_TARGETS)

if ANSWER_TARGETS is not None:
    example = candidate_latents_for(next(iter(train_loader)))
    print("Batch target latents:", tuple(example.shape))

# 26. Benchmark JEPA Forward Pass

Measure GPU memory consumption and forward-pass execution time for one real training batch. Include the online encoder, predictor, and all candidate target encodings. Do not perform an optimizer update.

In [ ]:

import time
import torch

device = torch.device(CONFIG["device"])

online_encoder.train()
predictor.train()
target_encoder.eval()

# Fetch one batch
benchmark_batch = next(iter(train_loader))

context_ids = benchmark_batch["context_input_ids"].to(device)
context_mask = benchmark_batch["context_attention_mask"].to(device)

candidate_ids = benchmark_batch["candidate_input_ids"].to(device)
candidate_mask = benchmark_batch["candidate_attention_mask"].to(device)

if device.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

start_time = time.perf_counter()

# Online branch
context_latent = online_encoder(context_ids, context_mask)
predicted_latent = predictor(context_latent)

# Target branch
with torch.no_grad():
    candidate_latents = target_encoder(
        candidate_ids,
        candidate_mask,
    )

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed = time.perf_counter() - start_time

print("=" * 60)
print("JEPA FORWARD BENCHMARK")
print("=" * 60)

print("Device:", device)
print("Batch size:", context_ids.shape[0])
print("Actual candidate inputs:", candidate_ids.shape[0])
print("Context latent:", tuple(context_latent.shape))
print("Candidate latents:", tuple(candidate_latents.shape))
print(f"Forward time: {elapsed:.3f} seconds")

if device.type == "cuda":
    allocated = torch.cuda.memory_allocated() / (1024**3)
    reserved = torch.cuda.memory_reserved() / (1024**3)
    peak = torch.cuda.max_memory_allocated() / (1024**3)

    print(f"\nCurrent allocated memory: {allocated:.2f} GB")
    print(f"Reserved memory: {reserved:.2f} GB")
    print(f"Peak allocated memory: {peak:.2f} GB")
else:
    print("\nCUDA unavailable; GPU memory statistics skipped.")

print("\nFORWARD BENCHMARK COMPLETED")


# 27. Implement One JEPA Training Step

Run one complete training step using a real batch. Compute context and candidate latents, calculate the padding-aware KL loss, backpropagate through the online encoder and predictor, update their weights, and then update the target encoder using EMA.

In [ ]:

import time
import torch

def train_one_step(batch):
    online_encoder.train()
    predictor.train()
    target_encoder.eval()

    # --------------------------------------------------
    # 1. Move batch tensors to the GPU
    # --------------------------------------------------
    context_ids = batch["context_input_ids"].to(device)
    context_attention = batch["context_attention_mask"].to(device)

    candidate_ids = batch["candidate_input_ids"].to(device)
    candidate_attention = batch["candidate_attention_mask"].to(device)

    candidate_mask = batch["candidate_mask"].to(device)
    gold_probabilities = batch["gold_probabilities"].to(device)

    batch_size = context_ids.shape[0]
    max_candidates = candidate_mask.shape[1]
    latent_dim = CONFIG["latent_dim"]

    # --------------------------------------------------
    # 2. Reset gradients
    # --------------------------------------------------
    optimizer.zero_grad(set_to_none=True)

    # --------------------------------------------------
    # 3. Online branch: encode the complete context
    # --------------------------------------------------
    context_latent = online_encoder(
        context_ids,
        context_attention,
    )

    predicted_latent = predictor(context_latent)

    # --------------------------------------------------
    # 4-5. Target branch: the candidates' target latents [batch, candidates, latent_dim]
    #      (section 25b: the fixed answer vectors, or the EMA-encoded candidate texts)
    # --------------------------------------------------
    candidate_latents = candidate_latents_for(batch)

    # --------------------------------------------------
    # 6. Calculate padding-aware candidate distribution loss
    # --------------------------------------------------
    loss, predicted_probabilities, distances, logits = (
        jepa_candidate_distribution_loss(
            predicted_latent=predicted_latent,
            candidate_target_latents=candidate_latents,
            gold_probabilities=gold_probabilities,
            temperature=CONFIG["temperature"],
            candidate_mask=candidate_mask,
        )
    )

    # --------------------------------------------------
    # 7. Backpropagate through online encoder + predictor
    # --------------------------------------------------
    loss.backward()

    gradient_norm = torch.nn.utils.clip_grad_norm_(
        list(online_encoder.parameters())
        + list(predictor.parameters()),
        max_norm=CONFIG["grad_clip_norm"],
    )

    # --------------------------------------------------
    # 8. Update online encoder and predictor
    # --------------------------------------------------
    optimizer.step()

    # --------------------------------------------------
    # 9. Update target encoder using EMA
    # --------------------------------------------------
    # Only the first design has a target encoder to follow; the answer targets are fixed.
    if CONFIG["target_mode"] == "candidate_text":

        update_ema_target(
            online_encoder,
            target_encoder,
            decay=CONFIG["target_ema_decay"],
        )

    return {
        "loss": loss.item(),
        "predicted_probabilities": predicted_probabilities.detach(),
        "gold_probabilities": gold_probabilities.detach(),
        "candidate_mask": candidate_mask.detach(),
        "gradient_norm": float(gradient_norm),
        "context_latent_shape": tuple(context_latent.shape),
        "candidate_latent_shape": tuple(candidate_latents.shape),
        "actual_candidate_count": int(candidate_mask.sum().item()),
    }


# ==================================================
# Execute one training step
# ==================================================

training_batch = next(iter(train_loader))

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

start_time = time.perf_counter()

step_result = train_one_step(training_batch)

if torch.cuda.is_available():
    torch.cuda.synchronize()

elapsed_time = time.perf_counter() - start_time

print("JEPA training step completed")
print("-" * 50)

print(f"Loss: {step_result['loss']:.6f}")
print(f"Gradient norm: {step_result['gradient_norm']:.6f}")

print(
    f"Context latent shape: "
    f"{step_result['context_latent_shape']}"
)

print(
    f"Candidate latent shape: "
    f"{step_result['candidate_latent_shape']}"
)

print(
    f"Actual candidate count: "
    f"{step_result['actual_candidate_count']}"
)

print(f"Training step time: {elapsed_time:.3f} seconds")

print("\nGold vs predicted distributions:")

for i in range(len(step_result["gold_probabilities"])):
    count = int(step_result["candidate_mask"][i].sum().item())

    gold = step_result["gold_probabilities"][i, :count].cpu()
    predicted = step_result["predicted_probabilities"][i, :count].cpu()

    print(f"\nSample {i + 1}:")
    print("Gold:     ", gold.numpy().round(4))
    print("Predicted:", predicted.numpy().round(4))

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    peak_allocated = torch.cuda.max_memory_allocated() / (1024 ** 3)

    print(f"\nGPU allocated memory: {allocated:.2f} GB")
    print(f"GPU peak allocated memory: {peak_allocated:.2f} GB")



# 28. Implement Training and Validation Epochs

A training step updates the online encoder, predictor, and EMA target encoder. An epoch runs that step across the complete training dataset.

After each training epoch, evaluate the model on the validation dataset without updating any weights.

We will track:
- Mean training KL loss.
- Mean validation KL loss.
- Top-choice agreement: how often the model's highest-scoring candidate matches the candidate with the highest gold preference.

Top-choice agreement is a useful ranking diagnostic, while KL loss evaluates the complete soft preference distribution.


In [ ]:

import torch
import time

def run_training_epoch(loader):
    online_encoder.train()
    predictor.train()
    target_encoder.eval()

    total_loss = 0.0
    total_examples = 0

    start_time = time.perf_counter()

    for batch in loader:
        result = train_one_step(batch)

        batch_size = batch["context_input_ids"].shape[0]

        total_loss += result["loss"] * batch_size
        total_examples += batch_size

    elapsed = time.perf_counter() - start_time

    return {
        "loss": total_loss / total_examples,
        "examples": total_examples,
        "time_seconds": elapsed,
    }


@torch.no_grad()
def run_validation_epoch(loader):
    online_encoder.eval()
    predictor.eval()
    target_encoder.eval()

    total_loss = 0.0
    total_examples = 0

    correct_top_choices = 0
    total_top_choices = 0

    for batch in loader:

        # ----------------------------------------------
        # 1. Move batch tensors to device
        # ----------------------------------------------
        context_ids = batch["context_input_ids"].to(device)
        context_attention = batch["context_attention_mask"].to(device)

        candidate_ids = batch["candidate_input_ids"].to(device)
        candidate_attention = batch["candidate_attention_mask"].to(device)

        candidate_mask = batch["candidate_mask"].to(device)
        gold_probabilities = batch["gold_probabilities"].to(device)

        batch_size = context_ids.shape[0]
        max_candidates = candidate_mask.shape[1]
        latent_dim = CONFIG["latent_dim"]

        # ----------------------------------------------
        # 2. Online branch
        # ----------------------------------------------
        context_latent = online_encoder(
            context_ids,
            context_attention,
        )

        predicted_latent = predictor(context_latent)

        # ----------------------------------------------
        # 3-4. Target branch (section 25b)
        # ----------------------------------------------
        candidate_latents = candidate_latents_for(batch)

        # ----------------------------------------------
        # 5. Calculate validation loss
        # ----------------------------------------------
        loss, predicted_probabilities, distances, logits = (
            jepa_candidate_distribution_loss(
                predicted_latent=predicted_latent,
                candidate_target_latents=candidate_latents,
                gold_probabilities=gold_probabilities,
                temperature=CONFIG["temperature"],
                candidate_mask=candidate_mask,
            )
        )

        total_loss += loss.item() * batch_size
        total_examples += batch_size

        # ----------------------------------------------
        # 6. Top-choice agreement
        # ----------------------------------------------
        predicted_best = predicted_probabilities.argmax(dim=1)
        gold_best = gold_probabilities.argmax(dim=1)

        correct_top_choices += (
            predicted_best == gold_best
        ).sum().item()

        total_top_choices += batch_size

    return {
        "loss": total_loss / total_examples,
        "top_choice_agreement": (
            correct_top_choices / total_top_choices
        ),
        "examples": total_examples,
    }


# ==================================================
# Run one complete training epoch and validation
# ==================================================

if CONFIG["run_demo_epoch"]:

    print("Starting one training epoch...")
    print("-" * 50)

    train_metrics = run_training_epoch(train_loader)

    print("\nTraining results")
    print(f"Training loss: {train_metrics['loss']:.6f}")
    print(f"Training examples: {train_metrics['examples']}")
    print(f"Training time: {train_metrics['time_seconds']:.2f} seconds")

    print("\nRunning validation...")
    print("-" * 50)

    validation_metrics = run_validation_epoch(validation_loader)

    print("\nValidation results")
    print(f"Validation loss: {validation_metrics['loss']:.6f}")
    print(
        f"Top-choice agreement: "
        f"{validation_metrics['top_choice_agreement']:.4f}"
    )
    print(f"Validation examples: {validation_metrics['examples']}")

else:
    print("Demo epoch skipped (run_demo_epoch is False): section 29 trains from a clean start.")



# 29. Establish Baselines and Train JEPA for Three Epochs

Start again from the clean pretrained weights (the steps above changed them), calculate a uniform-distribution baseline on the validation set, then train JEPA for `num_epochs` epochs (3).

The training loop will:
- Log progress every `log_interval` batches.
- Record batch and epoch metrics in CSV files.
- Save a full recovery checkpoint every `checkpoint_interval` batches and at epoch boundaries.
- Resume from the latest checkpoint if the run restarts (same batches, same order).
- Save the best checkpoint based on validation KL loss.
- Evaluate each included question type independently.
- Track validation KL loss and top-choice agreement.

All checkpoints include the online encoder, EMA target encoder, predictor, optimizer, the answer targets and training metadata. The answer targets are built from the fresh start and then never change.


In [ ]:

import os
import csv
import random
import time
import math
from pathlib import Path
from datetime import datetime

import torch


# ============================================================
# 1. Configuration
# ============================================================

NUM_EPOCHS = CONFIG["num_epochs"]
LOG_INTERVAL = CONFIG["log_interval"]
CHECKPOINT_INTERVAL = CONFIG["checkpoint_interval"]
TEMPERATURE = CONFIG["temperature"]
QUESTION_TYPES = CONFIG["included_question_types"]

RUN_DIR = Path(CONFIG["run_dir"])
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
LOG_DIR = RUN_DIR / "logs"
LATEST_CHECKPOINT = CHECKPOINT_DIR / "latest_checkpoint.pt"
BEST_CHECKPOINT = CHECKPOINT_DIR / "best_checkpoint.pt"

# Start from the clean pretrained weights: sections 27-28 already trained the live model.
online_encoder, target_encoder, predictor, optimizer = build_clean_jepa()

# The answer targets come from this fresh start (its projection head), then stay fixed.
ANSWER_TARGETS = build_answer_targets(online_encoder)
report_answer_targets(ANSWER_TARGETS)

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

BATCH_LOG_PATH = LOG_DIR / "batch_metrics.csv"
EPOCH_LOG_PATH = LOG_DIR / "epoch_metrics.csv"

print("Training configuration")
print("-" * 55)
print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch size: {train_loader.batch_size}")
print(f"Training batches per epoch: {len(train_loader)}")
print(f"Logging interval: {LOG_INTERVAL} batches")
print(f"Checkpoint interval: {CHECKPOINT_INTERVAL} batches")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")
print(f"Log directory: {LOG_DIR}")


# ============================================================
# 2. Identify question types in each batch
# ============================================================

def get_question_types(batch):
    for key in ("question_types", "question_type", "types"):
        if key in batch:
            return [
                str(value).lower()
                for value in batch[key]
            ]

    raise KeyError(
        "Question types are missing from the batch. "
        f"Available keys: {list(batch.keys())}"
    )


# Check the collator output before starting expensive training.
sample_batch = next(iter(validation_loader))
sample_types = get_question_types(sample_batch)

print("\nQuestion type metadata check")
print("Batch types:", sample_types)


# ============================================================
# 3. Validation and uniform baseline
# ============================================================

@torch.no_grad()
def evaluate_jepa(loader):

    online_encoder.eval()
    predictor.eval()
    target_encoder.eval()

    metrics = {
        group: {
            "loss_sum": 0.0,
            "uniform_kl_sum": 0.0,
            "correct": 0,
            "count": 0,
        }
        for group in ["overall", *QUESTION_TYPES]
    }

    for batch in loader:

        context_ids = batch["context_input_ids"].to(device)
        context_attention = batch["context_attention_mask"].to(device)

        candidate_ids = batch["candidate_input_ids"].to(device)
        candidate_attention = batch["candidate_attention_mask"].to(device)

        candidate_mask = batch["candidate_mask"].to(device)
        gold = batch["gold_probabilities"].to(device)

        question_types = get_question_types(batch)

        batch_size = context_ids.shape[0]
        max_candidates = candidate_mask.shape[1]
        latent_dim = CONFIG["latent_dim"]

        # Online branch
        context_latent = online_encoder(
            context_ids,
            context_attention,
        )

        predicted_latent = predictor(context_latent)

        # Target branch (section 25b)
        candidate_latents = candidate_latents_for(batch)

        # Calculate model distribution
        _, predicted_probs, distances, logits = (
            jepa_candidate_distribution_loss(
                predicted_latent=predicted_latent,
                candidate_target_latents=candidate_latents,
                gold_probabilities=gold,
                temperature=TEMPERATURE,
                candidate_mask=candidate_mask,
            )
        )

        # Calculate metrics per sample
        for i in range(batch_size):

            count = int(candidate_mask[i].sum().item())

            p = gold[i, :count]
            q = predicted_probs[i, :count].clamp_min(1e-12)

            # KL(gold || predicted)
            sample_kl = (
                p * (p.clamp_min(1e-12).log() - q.log())
            ).sum().item()

            # Uniform baseline KL(gold || uniform)
            uniform_kl = (
                p * (p.clamp_min(1e-12).log() + math.log(count))
            ).sum().item()

            predicted_best = predicted_probs[i, :count].argmax().item()
            gold_best = p.argmax().item()

            question_type = question_types[i]

            if question_type not in QUESTION_TYPES:
                raise ValueError(
                    f"Unexpected question type: {question_type}"
                )

            for group in ("overall", question_type):

                metrics[group]["loss_sum"] += sample_kl
                metrics[group]["uniform_kl_sum"] += uniform_kl
                metrics[group]["correct"] += int(
                    predicted_best == gold_best
                )
                metrics[group]["count"] += 1

    # Final averages
    results = {}

    for group, values in metrics.items():

        count = values["count"]

        if count == 0:
            continue

        results[group] = {
            "kl_loss": values["loss_sum"] / count,
            "uniform_kl": values["uniform_kl_sum"] / count,
            "top_choice_agreement": values["correct"] / count,
            "examples": count,
        }

    return results


# ============================================================
# 4. Establish baseline before training
# ============================================================

print("\nCalculating initial validation baseline...")
print("-" * 55)

baseline_results = evaluate_jepa(validation_loader)

print("\nInitial validation metrics")

for group, values in baseline_results.items():

    print(f"\n{group.upper()} ({values['examples']} examples)")
    print(f"JEPA KL loss:       {values['kl_loss']:.6f}")
    print(f"Uniform KL baseline:{values['uniform_kl']:.6f}")
    print(
        f"Top-choice agreement: "
        f"{values['top_choice_agreement']:.4f}"
    )


# ============================================================
# 5. CSV logging setup
# ============================================================

batch_fields = [
    "timestamp",
    "epoch",
    "batch",
    "total_batches",
    "batch_loss",
    "running_loss",
    "batches_per_second",
    "gpu_allocated_gb",
    "gpu_peak_gb",
]

epoch_fields = [
    "epoch",
    "training_loss",
    "validation_kl",
    "validation_uniform_kl",
    "validation_top_choice",
    *[f"{t}_{m}" for t in QUESTION_TYPES for m in ("kl", "top_choice")],
    "training_seconds",
]

RESUMING = LATEST_CHECKPOINT.exists()

if not RESUMING:

    with open(BATCH_LOG_PATH, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=batch_fields)
        writer.writeheader()

    with open(EPOCH_LOG_PATH, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=epoch_fields)
        writer.writeheader()


# ============================================================
# 6. Checkpoint function
# ============================================================

def save_checkpoint(path, epoch, batch, best_validation_loss, history,
                    epoch_loss_sum=0.0, epoch_examples=0):

    checkpoint = {
        "epoch_loss_sum": epoch_loss_sum,
        "epoch_examples": epoch_examples,
        "epoch": epoch,
        "batch": batch,
        "best_validation_loss": best_validation_loss,

        "online_encoder": online_encoder.state_dict(),
        "target_encoder": target_encoder.state_dict(),
        "predictor": predictor.state_dict(),
        "answer_targets": (
            {k: v.cpu() for k, v in ANSWER_TARGETS.items()} if ANSWER_TARGETS else None
        ),

        "optimizer": optimizer.state_dict(),

        "config": {k: str(v) if k == "device" else v for k, v in CONFIG.items()},
        "temperature": TEMPERATURE,
        "history": history,

        "python_rng_state": __import__("random").getstate(),
        "torch_rng_state": torch.get_rng_state(),

        "cuda_rng_state": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),
    }

    # Write to a temporary file first, so an interrupted save never leaves a broken checkpoint.
    temporary = path.with_suffix(".tmp")
    torch.save(checkpoint, temporary)
    os.replace(temporary, path)


# ============================================================
# 7. Three-epoch training loop
# ============================================================

best_validation_loss = float("inf")
training_history = []

total_batches = len(train_loader)

start_epoch, skip_batches = 1, 0
resumed_loss_sum, resumed_examples = 0.0, 0

if RESUMING:

    checkpoint = torch.load(LATEST_CHECKPOINT, map_location=device, weights_only=False)

    online_encoder.load_state_dict(checkpoint["online_encoder"])
    target_encoder.load_state_dict(checkpoint["target_encoder"])
    predictor.load_state_dict(checkpoint["predictor"])

    if checkpoint.get("answer_targets"):
        ANSWER_TARGETS = {k: v.to(device) for k, v in checkpoint["answer_targets"].items()}
    optimizer.load_state_dict(checkpoint["optimizer"])

    best_validation_loss = checkpoint["best_validation_loss"]
    training_history = checkpoint["history"]

    random.setstate(checkpoint["python_rng_state"])
    torch.set_rng_state(checkpoint["torch_rng_state"])
    if torch.cuda.is_available() and checkpoint["cuda_rng_state"] is not None:
        torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state"])

    if checkpoint["batch"] >= total_batches:
        start_epoch = checkpoint["epoch"] + 1              # that epoch finished
    else:
        start_epoch = checkpoint["epoch"]                  # continue inside it
        skip_batches = checkpoint["batch"]
        resumed_loss_sum = checkpoint["epoch_loss_sum"]
        resumed_examples = checkpoint["epoch_examples"]

    print(f"\nRESUMED from {LATEST_CHECKPOINT}: epoch {start_epoch}, "
          f"after batch {skip_batches}/{total_batches}")

print("\nStarting JEPA training")
print("=" * 65)

for epoch in range(start_epoch, NUM_EPOCHS + 1):

    online_encoder.train()
    predictor.train()
    target_encoder.eval()

    # The same order for this epoch on every attempt, so a resume can skip what was done.
    train_generator.manual_seed(CONFIG["seed"] + epoch)

    skip = skip_batches if epoch == start_epoch else 0
    epoch_loss_sum = resumed_loss_sum if skip else 0.0
    epoch_examples = resumed_examples if skip else 0

    recent_losses = []
    epoch_start = time.perf_counter()
    last_log_time = epoch_start

    print(f"\nEPOCH {epoch}/{NUM_EPOCHS}")
    print("-" * 65)

    for batch_idx, batch in enumerate(train_loader, start=1):

        if batch_idx <= skip:
            continue

        result = train_one_step(batch)

        batch_size = batch["context_input_ids"].shape[0]

        loss_value = result["loss"]

        epoch_loss_sum += loss_value * batch_size
        epoch_examples += batch_size

        recent_losses.append(loss_value)

        # --------------------------------------------
        # Progress logging
        # --------------------------------------------
        if (
            batch_idx % LOG_INTERVAL == 0
            or batch_idx == total_batches
        ):

            now = time.perf_counter()

            interval = now - last_log_time
            batches_done = min(LOG_INTERVAL, batch_idx - skip)

            speed = batches_done / max(interval, 1e-9)

            running_loss = epoch_loss_sum / epoch_examples

            if torch.cuda.is_available():

                gpu_allocated = (
                    torch.cuda.memory_allocated() / (1024 ** 3)
                )

                gpu_peak = (
                    torch.cuda.max_memory_allocated() / (1024 ** 3)
                )

            else:
                gpu_allocated = 0.0
                gpu_peak = 0.0

            print(
                f"[{batch_idx:4d}/{total_batches}] "
                f"Batch loss: {loss_value:.4f} | "
                f"Running loss: {running_loss:.4f} | "
                f"{speed:.2f} batches/sec | "
                f"GPU: {gpu_allocated:.2f} GB"
            )

            with open(BATCH_LOG_PATH, "a", newline="") as f:

                writer = csv.DictWriter(
                    f,
                    fieldnames=batch_fields,
                )

                writer.writerow({
                    "timestamp": datetime.now().isoformat(),
                    "epoch": epoch,
                    "batch": batch_idx,
                    "total_batches": total_batches,
                    "batch_loss": loss_value,
                    "running_loss": running_loss,
                    "batches_per_second": speed,
                    "gpu_allocated_gb": gpu_allocated,
                    "gpu_peak_gb": gpu_peak,
                })

            last_log_time = now

        # --------------------------------------------
        # Periodic recovery checkpoint
        # --------------------------------------------
        if batch_idx % CHECKPOINT_INTERVAL == 0:

            save_checkpoint(
                LATEST_CHECKPOINT,
                epoch,
                batch_idx,
                best_validation_loss,
                training_history,
                epoch_loss_sum,
                epoch_examples,
            )

            print(
                f"  Recovery checkpoint saved "
                f"at batch {batch_idx}"
            )

    # --------------------------------------------
    # End of training epoch
    # --------------------------------------------
    training_loss = epoch_loss_sum / epoch_examples
    training_seconds = time.perf_counter() - epoch_start

    print("\nTraining epoch completed")
    print(f"Training loss: {training_loss:.6f}")
    print(f"Training time: {training_seconds / 60:.2f} minutes")

    # --------------------------------------------
    # Validation
    # --------------------------------------------
    print("\nRunning validation...")

    validation_results = evaluate_jepa(validation_loader)

    overall = validation_results["overall"]

    print("\nValidation results")
    print("-" * 55)

    for group, values in validation_results.items():

        print(
            f"{group.upper():8s} | "
            f"KL: {values['kl_loss']:.6f} | "
            f"Uniform KL: {values['uniform_kl']:.6f} | "
            f"Top-choice: {values['top_choice_agreement']:.4f} | "
            f"N: {values['examples']}"
        )

    # --------------------------------------------
    # Save history
    # --------------------------------------------
    epoch_record = {
        "epoch": epoch,
        "training_loss": training_loss,
        "validation": validation_results,
        "training_seconds": training_seconds,
    }

    training_history.append(epoch_record)

    with open(EPOCH_LOG_PATH, "a", newline="") as f:

        writer = csv.DictWriter(
            f,
            fieldnames=epoch_fields,
        )

        writer.writerow({
            "epoch": epoch,
            "training_loss": training_loss,
            "validation_kl": overall["kl_loss"],
            "validation_uniform_kl": overall["uniform_kl"],
            "validation_top_choice": overall["top_choice_agreement"],
            **{
                f"{t}_{m}": validation_results[t][key]
                for t in QUESTION_TYPES if t in validation_results
                for m, key in (("kl", "kl_loss"), ("top_choice", "top_choice_agreement"))
            },
            "training_seconds": training_seconds,
        })

    # --------------------------------------------
    # Save best validation checkpoint
    # --------------------------------------------
    if overall["kl_loss"] < best_validation_loss:

        best_validation_loss = overall["kl_loss"]

        save_checkpoint(
            BEST_CHECKPOINT,
            epoch,
            total_batches,
            best_validation_loss,
            training_history,
        )

        print(
            f"\nNew best checkpoint saved. "
            f"Validation KL: {best_validation_loss:.6f}"
        )

    # --------------------------------------------
    # Save latest full checkpoint (marks the epoch as finished)
    # --------------------------------------------
    save_checkpoint(
        LATEST_CHECKPOINT,
        epoch,
        total_batches,
        best_validation_loss,
        training_history,
    )

    print("\nEpoch summary")
    print(f"Epoch: {epoch}/{NUM_EPOCHS}")
    print(f"Training loss: {training_loss:.6f}")
    print(f"Validation loss: {overall['kl_loss']:.6f}")
    print(f"Best validation loss: {best_validation_loss:.6f}")
    print("=" * 65)


# ============================================================
# 8. Final summary
# ============================================================

print(f"\n{NUM_EPOCHS}-EPOCH TRAINING COMPLETED")
print("=" * 65)

print(f"Best validation KL: {best_validation_loss:.6f}")

print("\nTraining history:")

for record in training_history:

    validation = record["validation"]["overall"]

    print(
        f"Epoch {record['epoch']}: "
        f"Train={record['training_loss']:.6f} | "
        f"Validation={validation['kl_loss']:.6f} | "
        f"Top-choice={validation['top_choice_agreement']:.4f}"
    )

print("\nGenerated files:")

for path in [
    LATEST_CHECKPOINT,
    BEST_CHECKPOINT,
    BATCH_LOG_PATH,
    EPOCH_LOG_PATH,
]:

    print(f"{path} | Exists: {path.exists()}")

print("\nCheckpoint and log directory:")
print(RUN_DIR.resolve())


# 30. Benchmark on Typed Decisions (included question types only)

Score the best checkpoint on the **Typed Decisions test split** (`LocalLLaMA/typed-decisions`, 400 records), using only the question types in `included_question_types` (this run: the 600 NOUL questions). The test split is never used for training or selection.

Each question is built exactly as in training: the same state + question + candidates for the online input, one "Selected candidate" input per candidate for the EMA target, and the same candidate rule (a NOUL question without criteria gets `true` / `false`).

Measures, per question then averaged:
- **Accuracy:** the top predicted candidate is the gold's top candidate (the measure behind Signpost-D-4B's 0.712, there over all 2,000 questions).
- **KL** (gold ‖ predicted) and **Brier**.

Reference rows on the same questions:
- **Signpost-D-4B:** its public test predictions.
- **Uniform:** equal probability for every candidate.
- **Majority:** always the candidate that is most often the gold's top in our training split.

This is a NOUL-only comparison, not a Typed Decisions score.

In [ ]:
import json
import math
from collections import Counter, defaultdict

import pandas as pd
from huggingface_hub import hf_hub_download
from torch.utils.data import DataLoader

QUESTION_TYPES = CONFIG["included_question_types"]

# --------------------------------------------------
# 1. Load the best checkpoint
# --------------------------------------------------

best = torch.load(BEST_CHECKPOINT, map_location=device, weights_only=False)

online_encoder.load_state_dict(best["online_encoder"])
target_encoder.load_state_dict(best["target_encoder"])
predictor.load_state_dict(best["predictor"])

if best.get("answer_targets"):
    ANSWER_TARGETS = {k: v.to(device) for k, v in best["answer_targets"].items()}

print(f"Target mode: {CONFIG['target_mode']}")

print(f"Best checkpoint: epoch {best['epoch']}, validation KL {best['best_validation_loss']:.6f}")

# --------------------------------------------------
# 2. Typed Decisions test questions, built as in training
# --------------------------------------------------

def parsed(value):
    return json.loads(value) if isinstance(value, str) else value


td_path = hf_hub_download(CONFIG["td_dataset"], CONFIG["td_file"], repo_type="dataset")
td_frame = pd.read_parquet(td_path)

td_samples = []

for case_index, row in enumerate(td_frame.to_dict("records")):

    state = parsed(row["state"])
    questions = parsed(row["questions"])
    gold = parsed(row["gold"])

    for question_id, question in questions.items():

        question_type = str(question.get("type", "")).lower()

        if question_type not in QUESTION_TYPES:
            continue

        candidates = get_candidates(question)              # the rule from section 3
        probabilities = {str(k): float(v) for k, v in gold[question_id]["probabilities"].items()}

        assert {c["key"] for c in candidates} == set(probabilities), (row["id"], question_id)

        td_samples.append({
            "record_id": row["id"],
            "case_index": case_index,
            "workflow": row["workflow"],
            "question_id": question_id,
            "question_type": question_type,
            "instructions": question.get("instructions"),
            "state": state,
            "candidates": candidates,
            "candidate_count": len(candidates),
            "gold_probabilities": probabilities,
        })

td_dataset = JEPADataset(td_samples, tokenizer, MAX_LENGTH)

longest = max(len(td_dataset[i]["online"]["input_ids"]) for i in range(len(td_dataset)))

print(f"Typed Decisions test: {len(td_frame)} records, {len(td_samples)} questions "
      f"({dict(Counter(s['question_type'] for s in td_samples))})")
print(f"Longest online input: {longest} tokens (limit {MAX_LENGTH})")

td_loader = DataLoader(
    td_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    collate_fn=jepa_collate_fn,
)

# --------------------------------------------------
# 3. Predict
# --------------------------------------------------

@torch.no_grad()
def predict_jepa(loader):

    online_encoder.eval()
    predictor.eval()
    target_encoder.eval()

    predictions = []

    for batch in loader:

        candidate_mask = batch["candidate_mask"].to(device)
        gold = batch["gold_probabilities"].to(device)

        predicted_latent = predictor(online_encoder(
            batch["context_input_ids"].to(device),
            batch["context_attention_mask"].to(device),
        ))

        candidate_latents = candidate_latents_for(batch)

        _, probabilities, _, _ = jepa_candidate_distribution_loss(
            predicted_latent=predicted_latent,
            candidate_target_latents=candidate_latents,
            gold_probabilities=gold,
            temperature=CONFIG["temperature"],
            candidate_mask=candidate_mask,
        )

        for i, keys in enumerate(batch["candidate_keys"]):
            predictions.append({
                "record_id": batch["record_ids"][i],
                "question_id": batch["question_ids"][i],
                "probabilities": {k: float(probabilities[i, j]) for j, k in enumerate(keys)},
            })

    return predictions


jepa_predictions = predict_jepa(td_loader)

# --------------------------------------------------
# 4. Measures
# --------------------------------------------------

def top_key(probabilities):
    return max(probabilities, key=probabilities.get)


def measures(rows):
    """rows: [(gold {key: p}, predicted {key: p})]"""
    accuracy, kl, brier = [], [], []
    for g, p in rows:
        accuracy.append(top_key(p) == top_key(g))
        kl.append(sum(gv * math.log(gv / max(p[k], 1e-9)) for k, gv in g.items() if gv > 0))
        brier.append(sum((p[k] - g[k]) ** 2 for k in g))
    n = len(rows)
    return {"n": n, "accuracy": round(sum(accuracy) / n, 4),
            "kl": round(sum(kl) / n, 4), "brier": round(sum(brier) / n, 4)}


gold_of = {(s["record_id"], s["question_id"]): s["gold_probabilities"] for s in td_samples}
type_of = {(s["record_id"], s["question_id"]): s["question_type"] for s in td_samples}
workflow_of = {(s["record_id"], s["question_id"]): s["workflow"] for s in td_samples}

systems = {}

systems["JEPA (this run)"] = {
    (p["record_id"], p["question_id"]): p["probabilities"] for p in jepa_predictions
}

systems["Uniform"] = {
    key: {k: 1.0 / len(g) for k in g} for key, g in gold_of.items()
}

majority = Counter(
    top_key(s["gold_probabilities"]) for s in train_samples if s["question_type"] in QUESTION_TYPES
).most_common(1)[0][0]

systems[f"Majority ('{majority}')"] = {
    key: {k: float(k == majority) * 0.98 + 0.01 for k in g} for key, g in gold_of.items()
}

# Signpost-D-4B's public predictions: rows in the test split's order, NOUL labels yes/no.
reference_path = hf_hub_download(CONFIG["reference_model"], CONFIG["reference_file"])
reference = {}

for p in json.load(open(reference_path, encoding="utf-8")):
    key = (td_frame.iloc[p["case"]]["id"], p["question"])
    if key in gold_of:
        probabilities = p["probabilities"]
        if p["type"] == "noul":
            probabilities = {"true": probabilities["yes"], "false": probabilities["no"]}
        reference[key] = probabilities

if len(reference) == len(gold_of):
    systems["Signpost-D-4B (reference)"] = reference

# --------------------------------------------------
# 5. Report
# --------------------------------------------------

report = {}

for name, predicted in systems.items():
    rows = [(gold_of[key], predicted[key]) for key in gold_of]
    report[name] = {"all": measures(rows)}
    for t in QUESTION_TYPES:
        report[name][t] = measures([(gold_of[k], predicted[k]) for k in gold_of if type_of[k] == t])

print("\n" + "=" * 70)
print(f"TYPED DECISIONS TEST: {', '.join(QUESTION_TYPES).upper()} QUESTIONS ONLY")
print("=" * 70)

display(pd.DataFrame({name: r["all"] for name, r in report.items()}).T)

by_workflow = defaultdict(list)
for key, g in gold_of.items():
    by_workflow[workflow_of[key]].append((g, systems["JEPA (this run)"][key]))

print("\nJEPA accuracy by workflow")
display(pd.DataFrame({w: measures(rows) for w, rows in sorted(by_workflow.items())}).T)

with open(RUN_DIR / "td_benchmark.json", "w", encoding="utf-8") as f:
    json.dump({"config": {k: str(v) for k, v in CONFIG.items()},
               "best_epoch": best["epoch"],
               "report": report,
               "predictions": jepa_predictions}, f, indent=1)

print(f"\nSaved: {RUN_DIR / 'td_benchmark.json'}")
